## Common preprocessing

In [57]:
# -*- coding: utf-8 -*-
from bs4 import BeautifulSoup, NavigableString, Tag
from pathlib import Path
import html, re

class SamsungAuditHTMLPreprocessor:
    """
    삼성전자 감사보고서 HTML 전처리기 (평탄화 → 섹션 래핑 → 규칙적 드롭/정규화)
    - 요구사항별 메서드로 기능 분리
    """

    # ---------- 구성/상수 ----------
    SKIP_TAGS   = {"script","style","noscript"}
    KEEP_TAGS   = {"table","h1","h2","h3","h4","h5","h6"}
    BLOCK_BREAK = {"p","div","li","ul","ol","section","article","header","footer",
                   "aside","address","pre","blockquote","tr"}
    SECTION_NAMES = {1:"감사보고서", 2:"재무제표", 3:"주석", 4:"감사의견"}
    STOP_PHRASE = "외부감사 실시내용"

    # 숫자/텍스트 정규식
    RE_PARENS_NUM       = re.compile(r"^\(\s*([\d,\.]+)\s*\)$")
    RE_NUM_WITH_COMMAS  = re.compile(r"(?<=\d),(?=\d)")
    RE_DASH_ONLY        = re.compile(r"^\s*[–—-]+\s*$")

    # '계속' 제거용
    RE_P_CONT_HEADING = re.compile(
        r'^\s*["“”]?\d{1,3}(?:\s*[.)])?\s*[^:\n]*?,?\s*계\s*속\s*[:：;]?\s*["“”]?\s*$'
    )
    RE_SOLO_CONT_LINE = re.compile(
        r'^\s*["“”\']?계\s*속\s*[:：;]?\s*["“”\']?\s*$'
    )

    # ---------- 유틸 ----------
    @staticmethod
    def _normalize_ws(s: str) -> str:
        s = (s or "").replace("\xa0"," ")
        s = re.sub(r"[ \t]+", " ", s)
        return s.strip()

    @staticmethod
    def _has_section_class(tag: Tag) -> bool:
        classes = [str(c).lower() for c in (tag.get("class") or [])]
        return any("section" in c for c in classes)

    @staticmethod
    def _has_class_table(tag: Tag) -> bool:
        classes = [(c or "").lower() for c in (tag.get("class") or [])]
        return any(c == "table" or "table" in c for c in classes)

    # ---------- 0. 파싱/스킵 ----------
    def remove_skip_tags(self, soup: BeautifulSoup):
        for t in soup.find_all(list(self.SKIP_TAGS)):
            t.decompose()

    # ---------- 1. 숫자 정규화 ----------
    def normalize_table_numbers(self, soup: BeautifulSoup):
        """class에 'table' 포함된 테이블만 숫자 정규화"""
        for tbl in soup.find_all("table"):
            if not self._has_class_table(tbl):
                continue
            rows = tbl.find_all("tr", recursive=True)
            for r_idx, tr in enumerate(rows):
                cells = tr.find_all(["th","td"], recursive=False)
                for c_idx, td in enumerate(cells):
                    if r_idx == 0 or c_idx == 0 or td.name.lower() == "th":
                        continue
                    s = self._get_cell_text(td)
                    if not s:
                        continue
                    if self.RE_DASH_ONLY.match(s):
                        self._set_cell_text(td, "NA"); continue
                    m = self.RE_PARENS_NUM.match(s)
                    if m:
                        num = self.RE_NUM_WITH_COMMAS.sub("", m.group(1))
                        self._set_cell_text(td, f"-{num}"); continue
                    new_s = self.RE_NUM_WITH_COMMAS.sub("", s)
                    if new_s != s:
                        self._set_cell_text(td, new_s)

    def normalize_text_numbers_outside_tables(self, soup: BeautifulSoup):
        """테이블 외 텍스트에서 숫자 쉼표 제거 (1,234 → 1234)"""
        for tag in soup.find_all(True):
            if tag.name in self.SKIP_TAGS:
                continue
            for child in list(tag.children):
                if isinstance(child, NavigableString):
                    if self._is_inside_class_table(tag):
                        continue
                    text = str(child)
                    new_text = self.RE_NUM_WITH_COMMAS.sub("", text)
                    if new_text != text:
                        child.replace_with(new_text)

    @staticmethod
    def _get_cell_text(cell: Tag) -> str:
        txt = cell.get_text(separator="", strip=True)
        txt = txt.replace("\xa0"," ")
        txt = re.sub(r"[ \t]+", " ", txt)
        return txt.strip()

    @staticmethod
    def _set_cell_text(cell: Tag, text: str):
        cell.clear()
        cell.append(text)

    def _is_inside_class_table(self, node: Tag) -> bool:
        t = node.find_parent("table")
        return bool(t and self._has_class_table(t))

    # ---------- 2. 평탄화(텍스트/태그 스트림) + '계속' 제거 ----------
    def _traverse_stream(self, node: Tag):
        for child in getattr(node, "children", []):
            if isinstance(child, NavigableString):
                txt = self._normalize_ws(str(child))
                if txt:
                    yield ("text", txt)
                continue
            if not isinstance(child, Tag):
                continue
            name = (child.name or "").lower()
            if name in self.SKIP_TAGS:
                continue
            if name == "br":
                yield ("text","\n"); continue
            # 비어있는 섹션 헤더는 무시
            if name in {"h1","h2","h3","h4","h5","h6"} and self._has_section_class(child):
                if not child.get_text(strip=True):
                    continue
            # 섹션 class 요소/핵심 보존 태그는 통째 keep
            if self._has_section_class(child) or name in self.KEEP_TAGS:
                yield ("keep", str(child)); continue
            # 그 외는 자식 순회
            yield from self._traverse_stream(child)
            if name in self.BLOCK_BREAK:
                yield ("text","\n")

    def flatten_to_linear(self, soup: BeautifulSoup):
        """텍스트/태그 선형화 + '계속' 라인 제거 + 공백 정리"""
        body = soup.body or soup
        tokens = list(self._traverse_stream(body))
        linear, buf = [], ""
        def flush_buf():
            nonlocal buf
            if not buf:
                return
            for line in buf.split("\n"):
                line = self._normalize_ws(line)
                if not line:
                    continue
                if self._is_continuation_p(line) or self._is_solo_continuation(line):
                    continue
                linear.append(("p", f"<p>{html.escape(line)}</p>"))
            buf = ""
        for kind, payload in tokens:
            if kind == "text":
                buf += payload
            else:
                flush_buf()
                linear.append(("keep", payload))
        flush_buf()
        return linear

    def _is_continuation_p(self, line: str) -> bool:
        s = self._normalize_ws(line).strip().strip('"“”')
        return bool(self.RE_P_CONT_HEADING.match(s))

    def _is_solo_continuation(self, line: str) -> bool:
        s = self._normalize_ws(line).strip()
        return bool(self.RE_SOLO_CONT_LINE.match(s))

    # ---------- 3. 헤더 트리밍(SEC1 직전 p 위는 모두 제거) ----------
    @staticmethod
    def _soup_first_elem(html_str: str):
        try:
            return BeautifulSoup(html_str, "html.parser").find(True, recursive=True)
        except Exception:
            return None

    def _has_section_in_outer_html(self, html_str: str) -> bool:
        el = self._soup_first_elem(html_str)
        if not el: return False
        classes = [c.lower() for c in (el.get("class") or [])]
        return any("section" in c for c in classes)

    def trim_head_at_prev_p_before_SEC1(self, linear):
        """
        첫 번째 섹션(SEC1)으로 래핑될 'section class 요소'를 찾고,
        그 바로 위에 있는 <p>의 인덱스를 찾아서 **그 p부터** 시작한다.
        => 그 p보다 위의 모든 내용은 제거. (요구사항)
        """
        # 아직 래핑 전이므로: section class가 있는 첫 'keep' 찾기
        first_sec_keep_idx = next(
            (i for i,(k,v) in enumerate(linear) if k=="keep" and self._has_section_in_outer_html(v)),
            None
        )
        if first_sec_keep_idx is None:
            return linear  # 섹션 없으면 그대로
        # 바로 앞쪽에서 가장 가까운 p를 찾는다
        prev_p_idx = None
        for i in range(first_sec_keep_idx-1, -1, -1):
            if linear[i][0] == "p":
                prev_p_idx = i; break
        start_idx = prev_p_idx if prev_p_idx is not None else first_sec_keep_idx
        return linear[start_idx:]

    # ---------- 4. 섹션 래핑 ----------
    def wrap_sections_linear(self, linear):
        out, sec_idx, sec_open = [], 0, False
        for k, v in linear:
            if k=="keep" and self._has_section_in_outer_html(v):
                if sec_open:
                    out.append((None, f"</SEC{sec_idx}>"))
                sec_idx += 1
                sec_name = self.SECTION_NAMES.get(sec_idx, f"섹션{sec_idx}")
                out.append((None, f'<SEC{sec_idx} name="{sec_name}">'))
                sec_open = True
                out.append((k, v))
            else:
                out.append((k, v))
        if sec_open:
            out.append((None, f"</SEC{sec_idx}>"))
        return out

    # ---------- 5. SEC2/SEC4 테이블 드롭 ----------
    @staticmethod
    def _is_table_html(s: str) -> bool:
        return isinstance(s, str) and s.lstrip().lower().startswith("<table")

    def _find_section_bounds(self, linear_pairs, sec_n: int):
        open_i = close_i = None
        open_tag = f"<SEC{sec_n} "
        close_tag = f"</SEC{sec_n}>"
        for i,(k,v) in enumerate(linear_pairs):
            if k is None and isinstance(v,str) and v.startswith(open_tag):
                open_i = i; break
        if open_i is None: return (None, None)
        for j in range(open_i+1, len(linear_pairs)):
            if linear_pairs[j][0] is None and isinstance(linear_pairs[j][1],str) and linear_pairs[j][1].strip()==close_tag:
                close_i = j; break
        return (open_i, close_i)

    def drop_first_n_tables_in_sec2(self, linear_pairs, n=5):
        open_i, close_i = self._find_section_bounds(linear_pairs, 2)
        if open_i is None or close_i is None:
            return linear_pairs
        before = linear_pairs[:open_i+1]
        inside = linear_pairs[open_i+1:close_i]
        after  = linear_pairs[close_i:]
        out, dropped = [], 0
        for k,v in inside:
            if dropped < n and self._is_table_html(v):
                dropped += 1
                continue
            out.append((k,v))
        return before + out + after

    def drop_all_tables_in_sec4(self, linear_pairs):
        open_i, close_i = self._find_section_bounds(linear_pairs, 4)
        if open_i is None or close_i is None:
            return linear_pairs
        before = linear_pairs[:open_i+1]
        inside = linear_pairs[open_i+1:close_i]
        after  = linear_pairs[close_i:]
        out = [(k,v) for (k,v) in inside if not self._is_table_html(v)]
        return before + out + after

    # ---------- 6. 테일 컷 ----------
    def cut_tail_at_phrase(self, linear, phrase=None):
        phrase = phrase or self.STOP_PHRASE
        for i,(k,v) in enumerate(linear):
            if k=="p":
                txt = BeautifulSoup(v, "html.parser").get_text()
                if phrase in txt:
                    return linear[:i]
        return linear

    # ---------- 7. 저장 ----------
    @staticmethod
    def write_linear_to_html(linear, out_path: Path):
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path,"w",encoding="utf-8") as f:
            f.write('<!doctype html><meta charset="utf-8"><title>Preprocessed</title>'
                    '<style>body{font-family:system-ui,-apple-system,Segoe UI,Roboto,Apple SD Gothic Neo,Malgun Gothic,sans-serif;line-height:1.5}'
                    'p{margin:0 0 .7em} table{margin:1em 0;border-collapse:collapse}'
                    'td,th{border:1px solid #ddd;padding:.4em}</style><body>\n')
            for k, v in linear:
                f.write(v + "\n")
            f.write("</body>")
        return out_path

    # ---------- 8. 파이프라인 ----------
    def process_file(self, in_path: Path, out_dir: Path=None, drop_n_sec2_tables=5) -> Path:
        soup = BeautifulSoup(in_path.read_bytes(), "lxml")
        self.remove_skip_tags(soup)
        self.normalize_table_numbers(soup)
        self.normalize_text_numbers_outside_tables(soup)

        linear = self.flatten_to_linear(soup)
        # ★ 요구사항: SEC1 바로 위 p 기준으로 헤더 컷
        linear = self.trim_head_at_prev_p_before_SEC1(linear)

        linear = self.wrap_sections_linear(linear)
        linear = self.drop_first_n_tables_in_sec2(linear, n=drop_n_sec2_tables)
        linear = self.drop_all_tables_in_sec4(linear)
        linear = self.cut_tail_at_phrase(linear, phrase=self.STOP_PHRASE)

        out_dir = out_dir or (in_path.parent / "preprocessed")
        out_path = out_dir / f"{in_path.stem}_preprocess.html"
        self.write_linear_to_html(linear, out_path)
        return out_path

    def process_directory(self, dir_path: Path, patterns=("*.htm","*.html"), recursive=True, out_dir: Path=None, drop_n_sec2_tables=5):
        glober = dir_path.rglob if recursive else dir_path.glob
        files = []
        for pat in patterns:
            files.extend(sorted(glober(pat)))
        files = [p for p in files if p.is_file()]
        if not files:
            print(f"[WARN] No files in {dir_path} (patterns={patterns}, recursive={recursive})")
            return
        ok = fail = 0
        for fp in files:
            try:
                out_fp = self.process_file(fp, out_dir=out_dir, drop_n_sec2_tables=drop_n_sec2_tables)
                print(f"  ✓ {fp.name} -> {out_fp.relative_to(out_fp.parent)})")
                ok += 1
            except Exception as e:
                print(f"  ✗ {fp} -> {e}")
                fail += 1
        print(f"[DONE] 성공 {ok} / 실패 {fail}")

In [58]:
# 적용 예시 (Jupyter 셀/스크립트)
from pathlib import Path

pre = SamsungAuditHTMLPreprocessor()

# 1) 폴더 전체(재귀) 처리
INPUT_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024")  # ← 경로 수정
pre.process_directory(
    dir_path=INPUT_DIR,
    patterns=("*.htm","*.html"),
    recursive=True,
    out_dir=None,              # None이면 각 폴더 밑에 preprocessed/ 저장
    drop_n_sec2_tables=5       # SEC2에서 처음 N개 테이블 제거
)

  ✓ 감사보고서_2014.htm -> 감사보고서_2014_preprocess.html)
  ✓ 감사보고서_2015.htm -> 감사보고서_2015_preprocess.html)
  ✓ 감사보고서_2016.htm -> 감사보고서_2016_preprocess.html)
  ✓ 감사보고서_2017.htm -> 감사보고서_2017_preprocess.html)
  ✓ 감사보고서_2018.htm -> 감사보고서_2018_preprocess.html)
  ✓ 감사보고서_2019.htm -> 감사보고서_2019_preprocess.html)
  ✓ 감사보고서_2020.htm -> 감사보고서_2020_preprocess.html)
  ✓ 감사보고서_2021.htm -> 감사보고서_2021_preprocess.html)
  ✓ 감사보고서_2022.htm -> 감사보고서_2022_preprocess.html)
  ✓ 감사보고서_2023.htm -> 감사보고서_2023_preprocess.html)
  ✓ 감사보고서_2024.htm -> 감사보고서_2024_preprocess.html)
[DONE] 성공 11 / 실패 0


---

## Section 1 : 핵심감사사항, 이유 파싱

### VER2. 서술어

In [ ]:
# -*- coding: utf-8 -*-
from bs4 import BeautifulSoup
from pathlib import Path
import re, json

class SEC1AuditorKAMExporter:
    # ---- 정규식/목록 (KAM 바깥 오인 방지를 위해 보수적으로) ----
    RE_TITLE_ENUM   = re.compile(r"^\s*[·\-\u00B7]?\s*([가-힣A-Za-z])\.\s*(.+?)\s*$")  # 가./나./A.
    RE_TITLE_PLAIN  = re.compile(r"^(?=.{2,60}$)(?!.*[다요]\.$)(?!.*습니다\.$)(?!.*합니다\.$).+")
    RE_REASON_HEAD  = re.compile(r"핵심\s*감사\s*사항.*결정한\s*이유")
    RE_METHOD_HEAD  = re.compile(r"핵심\s*감사\s*사항.*다루어진\s*방법")
    RE_KAM_INTRO    = re.compile(r"^\s*핵심\s*감사\s*사항\s*$")
    RE_OTHER_HEAD   = re.compile(r"(경영진|지배기구|감사인).*책임|기타사항")
    RE_NOTE         = re.compile(r"주석\s*([0-9]+(?:\.[0-9]+)?)")  # 주석 2.23 / 주석 3 / 주석2(...)

    NON_KAM_HEADS = {
        "독립된 감사인의 감사보고서",
        "감사의견", "감사의견근거",
        "감사보고서", "재무제표", "주주 및 이사회 귀중",
        "재무제표감사에 대한 감사인의 책임",
        "재무제표에 대한 경영진과 지배기구의 책임",
        "기타사항"
    }

    # 조사 집합(보존용)
    PARTICLES = ("는","은","가","이","를","을","의","에","와","과","로","도","만","에서","에게","부터","까지","로서","로써")

    def __init__(self, company="삼성전자", parser="lxml", verbose=False):
        self.company = company
        self.parser  = parser  # "lxml" 권장
        self.verbose = verbose

        # 치환용 정규식 (조사 보존 + 뒤에 공백/문장부호/끝만 허용)
        pp = "|".join(map(re.escape, self.PARTICLES))
        # '회사' 뒤에 조사 0~1개 & 다음이 공백/비문자/문장끝
        self.RE_COMPANY_WORD = re.compile(rf"(회사)(?P<pp>{pp})?(?=\s|[^\w가-힣]|$)")
        # '우리' 뒤에 조사 0~1개 & 다음이 공백/비문자/문장끝  (우리나라 등은 건너뜀)
        self.RE_URI_WORD     = re.compile(rf"(우리)(?P<pp>{pp})?(?=\s|[^\w가-힣]|$)")

    # ------------ 유틸 ------------
    @staticmethod
    def _clean(s: str) -> str:
        return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()

    @staticmethod
    def _strip_all_spaces(s: str) -> str:
        return re.sub(r"\s+", "", s or "")

    def _load(self, p: Path) -> str:
        return p.read_text(encoding="utf-8", errors="ignore")

    def _year_from_name(self, path: Path) -> int | None:
        m = re.search(r"(20\d{2})", path.name)
        return int(m.group(1)) if m else None

    # ------------ 앵커/슬러그 헬퍼 (★ 유지) ------------
    @staticmethod
    def _slugify_basic(s: str) -> str:
        s = (s or "").strip().lower()
        s = re.sub(r"[^\w\s-]", "", s)
        s = re.sub(r"\s+", "-", s)
        s = re.sub(r"-{2,}", "-", s).strip("-")
        return s or "x"

    def _anchors(self, year: int):
        company_anchor = self._slugify_basic(self.company)
        year_anchor = f"{company_anchor}-{year}"
        return company_anchor, year_anchor

    @staticmethod
    def _note_refs_to_doc_ids(year: int, refs):
        out = []
        for r in refs or []:
            r = str(r).strip()
            if r:
                out.append(f"sec3-{year}-note-{r}")  # "2.23"도 그대로 허용
        return out

    # ------------ 한글 조사(이/가) 보정 (★ 추가) ------------
    @staticmethod
    def _has_jong(last_char: str) -> bool:
        code = ord(last_char)
        if 0xAC00 <= code <= 0xD7A3:
            return (code - 0xAC00) % 28 != 0
        return False

    @classmethod
    def _josa_iga(cls, word: str) -> str:
        if not word:
            return "이"
        last = word[-1]
        return "이" if cls._has_jong(last) else "가"

    # ------------ 텍스트 엔티티 치환 ------------
    def _normalize_entities(self, text: str, auditor_name: str) -> str:
        if not text:
            return text

        def rep_company(m):
            return self.company + (m.group("pp") or "")

        def rep_uri(m):
            return auditor_name + (m.group("pp") or "")

        # '회사' 먼저, 그다음 '우리'
        out = self.RE_COMPANY_WORD.sub(rep_company, text)
        out = self.RE_URI_WORD.sub(rep_uri, out)
        return out

    # ------------ SEC1 / Auditor ------------
    def _find_sec1_and_auditor(self, html_text: str):
        soup = BeautifulSoup(html_text, self.parser)
        sec1 = soup.find("sec1")
        auditor = ""

        if sec1:
            prev_p = sec1.find_previous("p")
            if prev_p:
                auditor = self._clean(prev_p.get_text(" ", strip=True))
        if not auditor:
            first_p = soup.find("p")
            if first_p:
                auditor = self._clean(first_p.get_text(" ", strip=True))

        auditor = self._strip_all_spaces(auditor)  # ★ 공백 모두 제거
        return (str(sec1) if sec1 else None), auditor

    # ------------ 주석 번호 파싱 ------------
    def _extract_note_refs(self, text: str):
        refs = []
        for m in self.RE_NOTE.finditer(text or ""):
            no = m.group(1)
            if no: refs.append(no.strip())
        # 중복 제거(순서 보존)
        seen, out = set(), []
        for x in refs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out

    # ------------ KAM 파서 ------------
    def _extract_kams_from_sec1(self, sec1_html: str, auditor_name: str):
        soup = BeautifulSoup(sec1_html, self.parser)
        ps = soup.find_all("p")

        items = []
        cur = None
        in_kam = False
        in_reason = False
        in_method = False

        def is_bullet(s: str) -> bool:
            return bool(re.match(r"^\s*[·\-\u00B7]", s))

        def looks_sentence(s: str) -> bool:
            # 문장 종결/긴 문장/마침표 등은 제목 아님(보수)
            return bool(re.search(r"[.?!]$|습니다$|합니다$|된다$|다\.$", s)) or (len(s) > 60)

        def commit():
            nonlocal cur
            if cur and cur.get("title"):
                title_norm  = self._normalize_entities(self._clean(cur["title"]), auditor_name)
                reason_text = "\n".join([self._clean(x) for x in cur.get("reason", []) if self._clean(x)])
                reason_norm = self._normalize_entities(reason_text, auditor_name)

                items.append({
                    "title": title_norm,
                    "reason": reason_norm,
                    "note_refs": self._extract_note_refs(reason_norm)
                })
            cur = None

        i = 0
        while i < len(ps):
            t = self._clean(ps[i].get_text(" ", strip=True))
            i += 1
            if not t:
                continue

            # 0) KAM 블록 진입/이탈
            if self.RE_KAM_INTRO.match(t):
                in_kam = True
                in_reason = False
                in_method = False
                if self.verbose: print("[KAM] enter")
                continue
            if self.RE_OTHER_HEAD.search(t):
                if in_kam and self.verbose: print("[KAM] exit by other head:", t)
                if in_kam:
                    commit()
                in_kam = False
                in_reason = False
                in_method = False
                continue
            if not in_kam:
                continue  # KAM 블록 밖은 모두 무시

            # 1) 방법 섹션 시작/유지 → 스킵
            if self.RE_METHOD_HEAD.search(t):
                in_method = True
                in_reason = False
                if self.verbose: print("[METHOD] enter")
                continue
            if in_method:
                pass  # 방법 내용은 추출하지 않음

            # 2) 이유 헤더 → 이유 누적 모드
            if self.RE_REASON_HEAD.search(t):
                in_reason = True
                in_method = False
                if cur is None:
                    cur = {"title":"", "reason":[]}
                if self.verbose: print("[REASON] start")
                continue

            # 3) 제목 탐지 (열거형 우선)
            m_enum = self.RE_TITLE_ENUM.match(t)
            if m_enum:
                if in_method and self.verbose: print("[METHOD] exit by new title")
                in_method = False
                commit()
                cur = {"title": m_enum.group(2).strip(), "reason":[]}
                in_reason = False
                if self.verbose: print("[TITLE] enum:", cur["title"])
                continue

            # 4) 평문 제목 후보 (보수적 + 룩어헤드로 이유 헤더 확인)
            if (t not in self.NON_KAM_HEADS) and (not is_bullet(t)) and (not looks_sentence(t)):
                nxt = self._clean(ps[i].get_text(" ", strip=True)) if i < len(ps) else ""
                if self.RE_REASON_HEAD.search(nxt):
                    if in_method and self.verbose: print("[METHOD] exit by new plain title")
                    in_method = False
                    commit()
                    cur = {"title": t, "reason":[]}
                    in_reason = False
                    if self.verbose: print("[TITLE] plain:", cur["title"])
                    continue

            # 5) 이유 본문 누적
            if in_reason and cur is not None:
                if is_bullet(t) or self.RE_METHOD_HEAD.search(t) or self.RE_OTHER_HEAD.search(t):
                    continue
                cur["reason"].append(t)
                continue

        commit()
        return items

    # ------------ 레코드 빌더 (앵커/노트ID 유지, document만 개선) ------------
    def _rec_auditor(self, year: int, auditor: str, src: str):
        ca, ya = self._anchors(year)
        iga = self._josa_iga(auditor)
        doc = f"{year}년 {self.company} 재무제표의 외부감사는 {auditor}{iga} 수행했습니다."
        return {
            "id": f"sec1-{year}-auditor",
            "document": doc,
            "metadata": {
                "company": self.company,
                "company_anchor": ca,
                "year": year,
                "year_anchor": ya,
                "section": "SEC1",
                "kind": "auditor",
                "group_id": f"sec1-{year}-root",
                "parent_id": f"sec1-{year}-root",
                "source": f"{src}#SEC1"
            }
        }

    def _rec_kam(self, year: int, idx: int, title: str, reason: str, notes, src: str):
        ca, ya = self._anchors(year)
        note_ids = self._note_refs_to_doc_ids(year, notes)
        # 한 줄 서술로 품질 개선 (방법 미포함)
        doc = f"핵심감사사항 #{idx}: {title}." + (f" [이유] {reason}" if reason else "")
        return {
            "id": f"sec1-{year}-kam-{idx}",
            "document": doc,
            "metadata": {
                "company": self.company,
                "company_anchor": ca,
                "year": year,
                "year_anchor": ya,
                "section": "SEC1",
                "kind": "kam",
                "kam_index": idx,
                "title": title,
                "reason": reason,
                "note_refs": list(notes or []),
                "note_doc_ids": note_ids,
                "group_id": f"sec1-{year}-root",
                "parent_id": f"sec1-{year}-root",
                "source": f"{src}#SEC1"
            }
        }

    def _rec_kam_none(self, year: int, src: str):
        ca, ya = self._anchors(year)
        return {
            "id": f"sec1-{year}-kam-none",
            "document": f"{year}년은 핵심감사사항 제도 도입 이전으로 핵심감사사항이 없습니다.",
            "metadata": {
                "company": self.company,
                "company_anchor": ca,
                "year": year,
                "year_anchor": ya,
                "section": "SEC1",
                "kind": "kam_none",
                "group_id": f"sec1-{year}-root",
                "parent_id": f"sec1-{year}-root",
                "source": f"{src}#SEC1"
            }
        }

    # ------------ 파일/디렉터리 추출 ------------
    def extract_file(self, path: Path):
        html = self._load(path)
        year = self._year_from_name(path)
        sec1_html, auditor = self._find_sec1_and_auditor(html)

        out = []
        if year:
            # Auditor
            if auditor:
                out.append(self._rec_auditor(year, auditor, path.name))

            # KAM
            if sec1_html:
                if year <= 2017:
                    out.append(self._rec_kam_none(year, path.name))
                else:
                    kams = self._extract_kams_from_sec1(sec1_html, auditor)
                    if kams:
                        for i, it in enumerate(kams, 1):
                            out.append(self._rec_kam(year, i, it["title"], it["reason"], it["note_refs"], path.name))
                    else:
                        out.append(self._rec_kam_none(year, path.name))
            else:
                if self.verbose: print(f"[INFO] {path.name}: <SEC1> not found")
        return out

    def extract_dir_to_jsonl(self, in_dir: Path, out_jsonl: Path,
                             patterns=("*.htm","*.html"), recursive=False):
        glober = in_dir.rglob if recursive else in_dir.glob
        files = []
        for pat in patterns:
            files.extend(sorted(glober(pat)))
        files = [p for p in files if p.is_file()]

        out_jsonl.parent.mkdir(parents=True, exist_ok=True)
        cnt = 0
        with open(out_jsonl, "w", encoding="utf-8") as f:
            for fp in files:
                try:
                    recs = self.extract_file(fp)
                    for r in recs:
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
                        cnt += 1
                except Exception as e:
                    print(f"[WARN] {fp} -> {e}")
        print(f"[DONE] {cnt} records -> {out_jsonl}")

In [35]:
from pathlib import Path
exporter = SEC1AuditorKAMExporter(company="삼성전자", parser="lxml")
INPUT_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
OUTPUT = INPUT_DIR / "sec1_auditor_kam_all.jsonl"
exporter.extract_dir_to_jsonl(INPUT_DIR, OUTPUT, patterns=("*_preprocess.html",), recursive=False)

[DONE] 26 records -> /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec1_auditor_kam_all.jsonl


---

## Section2. 5개 JSONl(재무상태표, 손익계산서, 포괄손익계산서, 자본변동표, 현금흐름표)

### 1.재무상태표

In [ ]:
# -*- coding: utf-8 -*-
"""
SEC2 (재무상태표) 파서 → JSONL 생성기 (수정 완성본)

- 변경점
  * 중간 섹션 총계(Ⅰ. 유동자산 / Ⅱ. 비유동자산 등)의 path를
    ["자산","유동자산"] 식으로 '중간 레벨'까지 포함하도록 수정
  * id: 숫자 접두어 존재 시 '-<숫자>-<라벨>'로 분리
  * metadata.line_item: 숫자 제거된 라벨, line_item_raw: 원문 보존
  * 2014년에만 prior(전기값) 유지
"""
from pathlib import Path
from bs4 import BeautifulSoup
import re, json, sys, os

# ===== 공통 설정 =====
SECTION = "SEC2"
STATEMENT_NAME = "재무상태표"
STATEMENT_CODE = "bs"
UNIT = "KRW_million"

# ===== 정규식 / 헬퍼 =====
RE_WS = re.compile(r"[ \t\u00A0]+")
RE_DASH_ONLY = re.compile(r"^\s*[–—-]\s*$")
RE_INT = re.compile(r"^-?\d+$")
RE_NOTESPLIT = re.compile(r"[,\s]+")
RE_ROMAN_PREFIX = re.compile(r"^\s*((?:[IVXLCDM]+|[\u2160-\u2188]+))\.\s*")
RE_NUM_PREFIX = re.compile(r"^\s*(\d+)[\.\)]?\s*")

ROMAN_ASCII_MAP = {
    "Ⅰ":"I","Ⅱ":"II","Ⅲ":"III","Ⅳ":"IV","Ⅴ":"V","Ⅵ":"VI","Ⅶ":"VII","Ⅷ":"VIII","Ⅸ":"IX","Ⅹ":"X",
    "Ⅺ":"XI","Ⅻ":"XII","Ⅼ":"L","Ⅽ":"C","Ⅾ":"D","Ⅿ":"M"
}

RE_TOP_KOR = re.compile(r"^(자\s*산|부\s*채|자\s*본)$")
HEADER_TOKENS = {"과목", "항목", "구분", "주석"}

def nrm(s: str) -> str:
    if s is None: return ""
    s = s.replace("\xa0", " ")
    return RE_WS.sub(" ", s).strip()

def collapse_hangul_letters(s: str) -> str:
    t = nrm(s)
    if not t: return t
    return re.sub(r"[ \t·\-\.\u00B7]+", "", t)

def strip_roman_prefix(s: str) -> str:
    return RE_ROMAN_PREFIX.sub("", s or "").strip()

def strip_num_prefix(s: str) -> str:
    return RE_NUM_PREFIX.sub("", s or "").strip()

def parse_line_number(s: str):
    if not s: return None
    m = RE_NUM_PREFIX.match(s)
    if not m: return None
    try:
        return int(m.group(1))
    except:
        return None

def clean_display_label(s: str) -> str:
    t = strip_roman_prefix(s or "")
    t = collapse_hangul_letters(t)
    return nrm(t)

def roman_to_ascii(raw: str) -> str:
    return "".join(ROMAN_ASCII_MAP.get(ch, ch) for ch in raw)

def roman_prefix(s: str):
    m = RE_ROMAN_PREFIX.match(s or "")
    if not m:
        return None, None
    raw = m.group(1)
    return raw, roman_to_ascii(raw)

def slugify(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[ ]+", "-", s)
    s = re.sub(r"[^a-z0-9\-가-힣]", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s[:140] or "x"

def parse_number(x: str):
    s = nrm(x).replace(",", "")
    if not s or s.upper()=="NA" or RE_DASH_ONLY.match(s): return None
    if RE_INT.match(s):
        try: return int(s)
        except: return None
    return None

def split_notes(txt: str):
    s = nrm(txt)
    if not s: return []
    parts = [p.strip("()[]") for p in RE_NOTESPLIT.split(s) if p]
    out = []
    for p in parts:
        p2 = p.replace(" ", "")
        if re.fullmatch(r"\d+(?:\.\d+)?", p2):
            out.append(p2)
    return out

def company_anchor(name: str) -> str:
    base = re.sub(r"[^\w\s-]", "", nrm(name)).strip().lower()
    base = re.sub(r"\s+", "-", base)
    return slugify(base) or "company"

def fmt_int(num: int | None) -> str:
    if num is None: return ""
    return f"{num:,}"

# ===== 순서(정렬) 기본값 =====
TOP_ORD = {"자산":1, "부채":2, "자본":3}
MID_ORD_BY_TOP_BASE = {
    "자산": {"유동자산":1, "비유동자산":2},
    "부채": {"유동부채":1, "비유동부채":2},
    "자본": {"자본":1, "지배기업소유주지분":1, "비지배지분":2}
}

# ===== 핵심 파서 =====
def extract_balance_sheet_records(html_path: Path, year: int, company: str):
    soup = BeautifulSoup(html_path.read_bytes(), "lxml")

    # thead에 '과목' & '당/전' 존재하는 첫 표
    target = None
    for t in soup.find_all("table"):
        thead = t.find("thead")
        if not thead: continue
        htxt = nrm(thead.get_text(" "))
        if ("과목" in htxt or "과 목" in htxt or "과  목" in htxt) and ("당" in htxt and "전" in htxt):
            target = t; break
    if target is None:
        return []

    comp_key = company_anchor(company)
    year_anchor = f"{comp_key}-{year}"
    statement_anchor = f"{year_anchor}-{STATEMENT_CODE}"
    sec_root_id = f"sec2-{year}-{STATEMENT_CODE}-root"

    current_top_label = None
    current_mid_label = None
    line_ord_counter = {}
    group_node_ids = {}

    mid_ord_dynamic = {k: dict(v) for k, v in MID_ORD_BY_TOP_BASE.items()}
    mid_next_idx = {}
    for top, m in mid_ord_dynamic.items():
        mid_next_idx[top] = (max(m.values()) + 1) if m else 1

    def make_node_id(code_path):
        key = "-".join(code_path) if code_path else "root"
        return f"sec2-{year}-{STATEMENT_CODE}-node-{slugify(key)}"

    def ensure_group_node(code_path):
        key = tuple(code_path)
        if key in group_node_ids: return group_node_ids[key]
        nid = make_node_id(code_path)
        group_node_ids[key] = nid
        return nid

    def make_record_id(code_path, line_item_disp):
        base = "-".join(code_path) if code_path else "root"
        num = parse_line_number(line_item_disp or "")
        li = strip_num_prefix(line_item_disp or "")
        if num is not None:
            return f"sec2-{year}-{STATEMENT_CODE}-{slugify(base + '-' + str(num) + '-' + li)}"
        else:
            return f"sec2-{year}-{STATEMENT_CODE}-{slugify(base + '-' + li)}"

    def is_total_line(label: str) -> bool:
        s = nrm(label).replace(" ", "")
        return ("총계" in s)

    def mid_ord(top_label: str | None, mid_label: str | None) -> int:
        if not top_label or not mid_label:
            return 99
        base = MID_ORD_BY_TOP_BASE.get(top_label, {})
        if mid_label in base:
            return base[mid_label]
        dyn = mid_ord_dynamic.setdefault(top_label, {})
        if mid_label not in dyn:
            dyn[mid_label] = mid_next_idx.setdefault(top_label, 1)
            mid_next_idx[top_label] = dyn[mid_label] + 1
        return dyn[mid_label]

    def get_path_ord(path_labels, line_item_display=None):
        out = []
        if path_labels:
            top = path_labels[0]
            out.append(TOP_ORD.get(top, 99))
        if len(path_labels) >= 2:
            top = path_labels[0]
            mid = path_labels[1]
            out.append(mid_ord(top, mid))
        if line_item_display is not None:
            num = parse_line_number(line_item_display or "")
            if num is not None:
                out.append(num)
            else:
                key = tuple(path_labels)
                line_ord_counter.setdefault(key, 0)
                line_ord_counter[key] += 1
                out.append(line_ord_counter[key])
        return out

    records = []
    rows = target.find_all("tr")
    prev_year_effective = (year == 2014)

    for tr in rows:
        cells = tr.find_all(["th","td"])
        if len(cells) < 2:
            continue

        first_txt = nrm(cells[0].get_text(" "))
        if first_txt.replace(" ", "") in HEADER_TOKENS:
            continue

        texts = [nrm(c.get_text(" ")) for c in cells]
        raw_label = texts[0] if len(texts)>=1 else ""
        raw_notes = texts[1] if len(texts)>=2 else ""

        # [label, notes, c1, c2, p1, p2] 패턴 우선
        cur = parse_number(texts[3]) if len(texts)>3 else None
        prv = parse_number(texts[5]) if len(texts)>5 else None
        if cur is None and len(texts)>2:
            alt = parse_number(texts[2]);  cur = alt if alt is not None else cur
        if prv is None and len(texts)>4:
            alt = parse_number(texts[4]);  prv = alt if alt is not None else prv
        if not prev_year_effective:
            prv = None

        display_label = clean_display_label(raw_label)
        display_label_no_space = display_label.replace(" ","")

        # ---- 최상위(자산/부채/자본) 라인 ----
        if RE_TOP_KOR.match(display_label_no_space):
            current_top_label = display_label_no_space
            current_mid_label = None
            ensure_group_node([current_top_label])
            if cur is None and prv is None and not is_total_line(raw_label):
                continue

        # ---- 로마 숫자 중간 그룹(Ⅰ. 유동자산 등) ----
        raw_roman, roman_ascii = roman_prefix(raw_label)
        is_mid_header = (raw_roman is not None)

        if is_mid_header:
            if current_top_label is None:
                tail = "자산" if display_label_no_space.endswith("자산") else \
                       ("부채" if display_label_no_space.endswith("부채") else "자본")
                current_top_label = tail
                ensure_group_node([current_top_label])

            current_mid_label = strip_roman_prefix(raw_label)
            current_mid_label = collapse_hangul_letters(current_mid_label)
            current_mid_label = nrm(current_mid_label)

            ensure_group_node([current_top_label, current_mid_label])

            # ★ 수정: 중간 섹션 자체가 총합/값이면 path에 중간 레벨 포함
            if (cur is not None) or (prv is not None) or is_total_line(raw_label):
                path_labels  = [current_top_label, current_mid_label]   # <-- 여기!
                path_codes   = path_labels[:]
                line_item_disp = current_mid_label

                rec_id = make_record_id(path_codes, line_item_disp)

                if cur is None and prv is None:
                    document = f"{year}년 {company}의 {path_labels[0]} 중, {path_labels[1]} 총합은 제공되지 않습니다."
                else:
                    if prv is not None:
                        document = f"{year}년 {company}의 {path_labels[0]} 중, {path_labels[1]} 총합은 {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                    else:
                        document = f"{year}년 {company}의 {path_labels[0]} 중, {path_labels[1]} 총합은 {fmt_int(cur)} 백만원입니다."

                note_refs = split_notes(raw_notes)
                values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
                path_ord = get_path_ord(path_labels, None)

                records.append({
                    "id": rec_id,
                    "document": document,
                    "metadata": {
                        "company": company, "year": year,
                        "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                        "unit": UNIT,
                        "path_labels": path_labels,
                        "path_codes": path_codes,
                        "path_ord": path_ord,
                        "line_item": current_mid_label,
                        "group_id": sec_root_id,
                        "parent_id": ensure_group_node(path_labels),
                        "year_anchor": year_anchor,
                        "statement_anchor": statement_anchor,
                        "note_refs": note_refs,
                        "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
                    },
                    "values": values,
                    "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
                })
                continue  # 다음 줄

        # ---- 총계 라인 ----
        if is_total_line(raw_label):
            lbl = display_label_no_space
            if "부채와자본총계" in lbl:
                path_labels = []
                parent_id = sec_root_id
            else:
                path_labels = [current_top_label] if current_top_label else []
                parent_id = ensure_group_node(path_labels) if path_labels else sec_root_id

            path_codes = path_labels[:]
            line_item = display_label_no_space
            rec_id = make_record_id(path_codes, line_item)

            if cur is None and prv is None:
                document = f"{year}년 {company}의 {line_item}는 제공되지 않습니다."
            else:
                if prv is not None:
                    document = f"{year}년 {company}의 {('/'.join(path_labels)+'의 ' if path_labels else '')}{line_item}는 {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                else:
                    document = f"{year}년 {company}의 {('/'.join(path_labels)+'의 ' if path_labels else '')}{line_item}는 {fmt_int(cur)} 백만원입니다."

            note_refs = split_notes(raw_notes)
            values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
            path_ord = get_path_ord(path_labels, None)

            records.append({
                "id": rec_id,
                "document": document,
                "metadata": {
                    "company": company, "year": year,
                    "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                    "unit": UNIT,
                    "path_labels": path_labels, "path_codes": path_codes,
                    "path_ord": path_ord,
                    "line_item": line_item,
                    "group_id": sec_root_id, "parent_id": parent_id,
                    "year_anchor": year_anchor, "statement_anchor": statement_anchor,
                    "note_refs": note_refs,
                    "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
                },
                "values": values,
                "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
            })
            continue

        # ---- 일반 라인(하위 항목) ----
        display_label = clean_display_label(raw_label)
        if not display_label:
            continue

        if current_mid_label:
            path_labels = [current_top_label, current_mid_label]
            parent_id   = ensure_group_node(path_labels)
        else:
            path_labels = [current_top_label] if current_top_label else []
            parent_id   = ensure_group_node(path_labels) if path_labels else sec_root_id

        path_codes = path_labels[:]
        line_item_raw = display_label
        base_item = strip_num_prefix(line_item_raw)
        line_idx = parse_line_number(line_item_raw)

        rec_id = make_record_id(path_codes, line_item_raw)

        if cur is None and prv is None:
            if len(path_labels) >= 2:
                document = f"{year}년 {company}의 {path_labels[0]}에서 {path_labels[1]} 중 하나인 {base_item}은(는) 제공되지 않습니다."
            elif len(path_labels) == 1:
                document = f"{year}년 {company}의 {path_labels[0]} 중 {base_item}은(는) 제공되지 않습니다."
            else:
                document = f"{year}년 {company}의 {base_item}은(는) 제공되지 않습니다."
        else:
            if len(path_labels) >= 2:
                if prv is not None:
                    document = f"{year}년의 {company} {path_labels[0]}에서 {path_labels[1]} 중 하나인 {base_item}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                else:
                    document = f"{year}년의 {company} {path_labels[0]}에서 {path_labels[1]} 중 하나인 {base_item}은(는) {fmt_int(cur)} 백만원입니다."
            elif len(path_labels) == 1:
                if prv is not None:
                    document = f"{year}년의 {company} {path_labels[0]} 중 {base_item}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                else:
                    document = f"{year}년의 {company} {path_labels[0]} 중 {base_item}은(는) {fmt_int(cur)} 백만원입니다."
            else:
                if prv is not None:
                    document = f"{year}년의 {company} {base_item}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                else:
                    document = f"{year}년의 {company} {base_item}은(는) {fmt_int(cur)} 백만원입니다."

        note_refs = split_notes(raw_notes)
        values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
        path_ord = get_path_ord(path_labels, line_item_raw)

        records.append({
            "id": rec_id,
            "document": document,
            "metadata": {
                "company": company, "year": year,
                "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                "unit": UNIT,
                "path_labels": path_labels,
                "path_codes": path_codes,
                "path_ord": path_ord,
                "line_item": base_item,
                "line_index": line_idx,
                "group_id": sec_root_id, "parent_id": parent_id,
                "year_anchor": year_anchor, "statement_anchor": statement_anchor,
                "note_refs": note_refs,
                "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
            },
            "values": values,
            "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
        })

    return records


# ===== 여러 파일을 하나의 JSONL로 누적 저장 =====
def process_many(
    in_dir: Path,
    year_by_name_func,
    out_jsonl: Path,
    company: str,
) -> Path:
    in_dir = Path(in_dir)
    out_jsonl = Path(out_jsonl)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    htmls = sorted([p for p in in_dir.rglob("*") if p.suffix.lower() in {".htm", ".html"}])

    with out_jsonl.open("w", encoding="utf-8") as fw:
        for hp in htmls:
            try:
                y = year_by_name_func(hp)
            except Exception:
                y = 2014
            recs = extract_balance_sheet_records(hp, y, company)

            recs.sort(key=lambda r: (
                r.get("metadata", {}).get("year", 0),
                tuple(r.get("metadata", {}).get("path_ord", [])),
                r.get("id", "")
            ))

            for rec in recs:
                fw.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return out_jsonl


# ===== 실행 스니펫(모든 연도 -> 하나의 JSONL) =====
def infer_year_from_name(p: Path) -> int:
    m = re.search(r"(19|20)\d{2}", p.stem)
    if not m:
        raise ValueError(f"연도 추출 실패: {p.name}")
    return int(m.group(0))

def run_folder_to_one_jsonl():
    IN_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
    OUT_JSONL = IN_DIR / "sec2_1_재무상태표.jsonl"

    test_file = IN_DIR / "감사보고서_2014.htm"
    if test_file.exists():
        _recs = extract_balance_sheet_records(test_file, 2014, "삼성전자")
        print(f"[TEST] 2014 레코드 수: {len(_recs)}")

    try:
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
    except OSError as e:
        print(f"[WARN] {e}; 홈으로 저장 시도")
        OUT_JSONL_HOME = Path.home() / "sec2_1_재무상태표.jsonl"
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL_HOME,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")

    try:
        with out_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                print(line.rstrip()[:200])
    except Exception:
        pass


# ===== CLI 엔트리 =====
if __name__ == "__main__":
    if len(sys.argv) >= 4:
        in_arg   = Path(sys.argv[1])
        out_jsonl = Path(sys.argv[2])
        company  = sys.argv[3]
        year     = int(sys.argv[4]) if len(sys.argv) >= 5 else None

        if in_arg.is_dir():
            def _infer_year(p: Path) -> int:
                m = re.search(r"(19|20)\d{2}", p.stem)
                return int(m.group(0)) if m else (year or 2014)

            out_path = process_many(
                in_arg,
                year_by_name_func=_infer_year,
                out_jsonl=out_jsonl,
                company=company,
            )
            print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
        else:
            y = year
            if y is None:
                m = re.search(r"(19|20)\d{2}", in_arg.stem)
                y = int(m.group(0)) if m else 2014
            recs = extract_balance_sheet_records(in_arg, y, company)
            out_jsonl.parent.mkdir(parents=True, exist_ok=True)
            with out_jsonl.open("w", encoding="utf-8") as fw:
                for rec in recs:
                    fw.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"[OK] Wrote: {out_jsonl}  ({out_jsonl.stat().st_size} bytes)")
    else:
        run_folder_to_one_jsonl()

[OK] Wrote: /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec2_balance_sheet_all_years.jsonl  (427455 bytes)
{"id": "sec2-2014-bs-root-부채와자본총계", "document": "2014년 삼성전자의 부채와자본총계는 164,060,583 백만원입니다(전기 154,825,957 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "재무상태표", "
{"id": "sec2-2014-bs-자산-자산총계", "document": "2014년 삼성전자의 자산의 자산총계는 164,060,583 백만원입니다(전기 154,825,957 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "재무상태표", "stat
{"id": "sec2-2014-bs-자산-유동자산-유동자산", "document": "2014년 삼성전자의 자산 중, 유동자산 총합은 62,054,773 백만원입니다(전기 60,603,694 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "재무상태표
{"id": "sec2-2014-bs-자산-유동자산-1-현금및현금성자산", "document": "2014년의 삼성전자 자산에서 유동자산 중 하나인 현금및현금성자산은(는) 1,643,318 백만원입니다(전기 2,030,307 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "
{"id": "sec2-2014-bs-자산-유동자산-2-단기금융상품", "document": "2014년의 삼성전자 자산에서

---

### 손익계산서

In [ ]:
# -*- coding: utf-8 -*-
"""
SEC2 (손익계산서) 파서 → JSONL 생성기

- 스키마: 재무상태표(bs)와 동일 (id/document/metadata/values/source)
- 규칙
  * 두 번째 <table class="table"> (또는 HEADER '과목/당/전' 탐지된 표들 중 2번째)을 우선 사용
  * 로마숫자 헤더(Ⅰ, Ⅱ, Ⅲ ...)는 중간 섹션으로 간주:
      - 해당 행이 값(또는 총합)을 가지면 그 자체를 레코드로 기록
      - 그 다음에 이어지는 하위 항목들은 path_labels=[중간섹션]에 소속
  * 숫자 접두어: id에는 -<숫자>-<항목>, metadata.line_item에는 숫자 제거
  * (추가) 괄호 속 단위 등은 line_item과 id에서 제거 (예: '(단위:원)')
  * 2014년에만 prior(전기값) 유지 (요구사항 일관)
"""
from pathlib import Path
from bs4 import BeautifulSoup
import re, json, sys, os

# ===== 공통 설정 =====
SECTION = "SEC2"
STATEMENT_NAME = "손익계산서"
STATEMENT_CODE = "is"                 # income statement
UNIT = "KRW_million"

# ===== 정규식 / 헬퍼 =====
RE_WS = re.compile(r"[ \t\u00A0]+")
RE_DASH_ONLY = re.compile(r"^\s*[–—-]\s*$")
RE_INT = re.compile(r"^-?\d+$")
RE_NOTESPLIT = re.compile(r"[,\s]+")
RE_ROMAN_PREFIX = re.compile(r"^\s*((?:[IVXLCDM]+|[\u2160-\u2188]+))\.\s*")
RE_NUM_PREFIX = re.compile(r"^\s*(\d+)[\.\)]?\s*")
RE_PARENS = re.compile(r"\([^)]*\)")  # () 안의 내용 전체

HEADER_TOKENS = {"과목", "항목", "구분", "주석"}

ROMAN_ASCII_MAP = {
    "Ⅰ":"I","Ⅱ":"II","Ⅲ":"III","Ⅳ":"IV","Ⅴ":"V","Ⅵ":"VI","Ⅶ":"VII","Ⅷ":"VIII","Ⅸ":"IX","Ⅹ":"X",
    "Ⅺ":"XI","Ⅻ":"XII","Ⅼ":"L","Ⅽ":"C","Ⅾ":"D","Ⅿ":"M"
}

def nrm(s: str) -> str:
    if s is None: return ""
    s = s.replace("\xa0", " ")
    return RE_WS.sub(" ", s).strip()

def collapse_hangul_letters(s: str) -> str:
    t = nrm(s)
    if not t: return t
    return re.sub(r"[ \t·\-\.\u00B7]+", "", t)

def strip_roman_prefix(s: str) -> str:
    return RE_ROMAN_PREFIX.sub("", s or "").strip()

def strip_num_prefix(s: str) -> str:
    return RE_NUM_PREFIX.sub("", s or "").strip()

def strip_parens(s: str) -> str:
    return nrm(RE_PARENS.sub("", s or ""))

def parse_line_number(s: str):
    if not s: return None
    m = RE_NUM_PREFIX.match(s)
    if not m: return None
    try:
        return int(m.group(1))
    except:
        return None

def clean_display_label(s: str) -> str:
    t = strip_roman_prefix(s or "")
    t = collapse_hangul_letters(t)
    return nrm(t)

def roman_prefix(s: str):
    m = RE_ROMAN_PREFIX.match(s or "")
    if not m:
        return None, None
    raw = m.group(1)
    ascii_ = "".join(ROMAN_ASCII_MAP.get(ch, ch) for ch in raw)
    return raw, ascii_

def slugify(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[ ]+", "-", s)
    s = re.sub(r"[^a-z0-9\-가-힣]", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s[:140] or "x"

def parse_number(x: str):
    s = nrm(x).replace(",", "")
    if not s or s.upper()=="NA" or RE_DASH_ONLY.match(s): return None
    if RE_INT.match(s):
        try: return int(s)
        except: return None
    return None

def split_notes(txt: str):
    s = nrm(txt)
    if not s: return []
    parts = [p.strip("()[]") for p in RE_NOTESPLIT.split(s) if p]
    out = []
    for p in parts:
        p2 = p.replace(" ", "")
        if re.fullmatch(r"\d+(?:\.\d+)?", p2):
            out.append(p2)
    return out

def company_anchor(name: str) -> str:
    base = re.sub(r"[^\w\s-]", "", nrm(name)).strip().lower()
    base = re.sub(r"\s+", "-", base)
    return slugify(base) or "company"

def fmt_int(num: int | None) -> str:
    if num is None: return ""
    return f"{num:,}"

# ===== 정렬 인덱스 =====
MID_ORD_BASE = {}  # 등장 순으로 동적 배정

# ===== 파서 =====
def extract_income_statement_records(html_path: Path, year: int, company: str):
    soup = BeautifulSoup(html_path.read_bytes(), "lxml")

    # class='table' 후보 + thead에 '과목'/'당'/'전' 있는 표들만
    tables = []
    for t in soup.find_all("table"):
        cls = " ".join((t.get("class") or [])).lower()
        if "table" not in cls:
            continue
        thead = t.find("thead")
        if not thead:
            continue
        htxt = nrm(thead.get_text(" "))
        if ("과목" in htxt or "과 목" in htxt or "과  목" in htxt) and ("당" in htxt and "전" in htxt):
            tables.append(t)

    target = None
    if len(tables) >= 2:
        target = tables[1]
    elif len(tables) == 1:
        target = tables[0]
    else:
        for t in soup.find_all("table"):
            thead = t.find("thead")
            if not thead: continue
            htxt = nrm(thead.get_text(" "))
            if ("과목" in htxt or "과 목" in htxt or "과  목" in htxt) and ("당" in htxt and "전" in htxt):
                target = t; break
    if target is None:
        return []

    comp_key = company_anchor(company)
    year_anchor = f"{comp_key}-{year}"
    statement_anchor = f"{year_anchor}-{STATEMENT_CODE}"
    sec_root_id = f"sec2-{year}-{STATEMENT_CODE}-root"

    current_mid_label = None
    line_ord_counter = {}
    group_node_ids = {}

    mid_ord_dynamic = dict(MID_ORD_BASE)
    mid_next_idx = (max(mid_ord_dynamic.values()) + 1) if mid_ord_dynamic else 1

    def ensure_group_node(mid_label):
        key = (mid_label,) if mid_label else tuple()
        if key in group_node_ids:
            return group_node_ids[key]
        if mid_label:
            nid = f"sec2-{year}-{STATEMENT_CODE}-node-{slugify(mid_label)}"
        else:
            nid = sec_root_id
        group_node_ids[key] = nid
        return nid

    def mid_ord(mid_label: str | None) -> int:
        if not mid_label:
            return 99
        nonlocal mid_next_idx
        if mid_label not in mid_ord_dynamic:
            mid_ord_dynamic[mid_label] = mid_next_idx
            mid_next_idx += 1
        return mid_ord_dynamic[mid_label]

    def get_path_ord(mid_label: str | None, line_item_display=None):
        out = []
        out.append(1)  # 상위 고정(정렬 안정용)
        if mid_label:
            out.append(mid_ord(mid_label))
        if line_item_display is not None:
            num = parse_line_number(line_item_display or "")
            if num is not None:
                out.append(num)
            else:
                key = (mid_label or "_root")
                line_ord_counter.setdefault(key, 0)
                line_ord_counter[key] += 1
                out.append(line_ord_counter[key])
        return out

    def make_record_id(mid_label: str | None, line_item_disp: str):
        """
        id 생성 시:
          - mid_label은 slug화
          - line_item_disp에서 숫자 접두 제거 → 괄호 내용 제거 → slug
        """
        base = "root" if not mid_label else slugify(mid_label)
        num = parse_line_number(line_item_disp or "")
        li  = strip_num_prefix(line_item_disp or "")
        li  = strip_parens(li)  # ★ 괄호 제거 추가
        if num is not None:
            return f"sec2-{year}-{STATEMENT_CODE}-{base}-{slugify(str(num)+'-'+li)}"
        else:
            return f"sec2-{year}-{STATEMENT_CODE}-{base}-{slugify(li)}"

    rows = target.find_all("tr")
    prev_year_effective = (year == 2014)
    records = []

    for tr in rows:
        cells = tr.find_all(["th","td"])
        if len(cells) < 2:
            continue

        first_txt = nrm(cells[0].get_text(" "))
        if first_txt.replace(" ", "") in HEADER_TOKENS:
            continue

        texts = [nrm(c.get_text(" ")) for c in cells]
        raw_label = texts[0] if len(texts)>=1 else ""
        raw_notes = texts[1] if len(texts)>=2 else ""

        # [label, notes, cur_l, cur_r, prv_l, prv_r]
        cur = parse_number(texts[3]) if len(texts)>3 else None
        prv = parse_number(texts[5]) if len(texts)>5 else None
        if cur is None and len(texts)>2:
            alt = parse_number(texts[2]);  cur = alt if alt is not None else cur
        if prv is None and len(texts)>4:
            alt = parse_number(texts[4]);  prv = alt if alt is not None else prv
        if not prev_year_effective:
            prv = None

        display_label = clean_display_label(raw_label)
        if not display_label:
            continue

        # 로마숫자 섹션?
        raw_roman, _ = roman_prefix(raw_label)
        is_mid_header = (raw_roman is not None)

        # --- 섹션 헤더 갱신 ---
        if is_mid_header:
            current_mid_label = clean_display_label(raw_label)
            ensure_group_node(current_mid_label)

            # 섹션행 자체가 값 보유 시 레코드
            if (cur is not None) or (prv is not None):
                path_labels = [current_mid_label]
                parent_id = ensure_group_node(current_mid_label)

                rec_id = make_record_id(current_mid_label, current_mid_label)
                note_refs = split_notes(raw_notes)
                values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
                path_ord = get_path_ord(current_mid_label, None)

                if prv is not None:
                    doc = f"{year}년 {company}의 {current_mid_label}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
                else:
                    doc = f"{year}년 {company}의 {current_mid_label}은(는) {fmt_int(cur)} 백만원입니다."

                records.append({
                    "id": rec_id,
                    "document": doc,
                    "metadata": {
                        "company": company, "year": year,
                        "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                        "unit": UNIT,
                        "path_labels": path_labels,
                        "path_codes": path_labels[:],
                        "path_ord": path_ord,
                        "line_item": current_mid_label,
                        "group_id": sec_root_id,
                        "parent_id": parent_id,
                        "year_anchor": year_anchor,
                        "statement_anchor": statement_anchor,
                        "note_refs": note_refs,
                        "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
                    },
                    "values": values,
                    "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
                })
            continue  # 다음 줄

        # --- 일반 라인 (섹션 내부 또는 루트) ---
        mid = current_mid_label  # 있을 수도, 없을 수도
        path_labels = [mid] if mid else []
        parent_id = ensure_group_node(mid)

        line_item_raw = display_label
        base_item = strip_num_prefix(line_item_raw)
        base_item_clean = strip_parens(base_item)  # ★ 괄호 제거
        line_idx = parse_line_number(line_item_raw)

        rec_id = make_record_id(mid, line_item_raw)  # 내부에서 괄호 제거 처리됨
        note_refs = split_notes(raw_notes)
        values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
        path_ord = get_path_ord(mid, line_item_raw)

        # 설명 문장에 정제본 사용
        item_for_doc = base_item_clean or base_item

        if mid:
            if prv is not None and cur is not None:
                doc = f"{year}년의 {company} {mid} 중 하나인 {item_for_doc}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
            elif cur is not None:
                doc = f"{year}년의 {company} {mid} 중 하나인 {item_for_doc}은(는) {fmt_int(cur)} 백만원입니다."
            else:
                doc = f"{year}년 {company}의 {mid} 중 하나인 {item_for_doc}은(는) 제공되지 않습니다."
        else:
            if prv is not None and cur is not None:
                doc = f"{year}년의 {company} {item_for_doc}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."
            elif cur is not None:
                doc = f"{year}년의 {company} {item_for_doc}은(는) {fmt_int(cur)} 백만원입니다."
            else:
                doc = f"{year}년 {company}의 {item_for_doc}은(는) 제공되지 않습니다."

        records.append({
            "id": rec_id,
            "document": doc,
            "metadata": {
                "company": company, "year": year,
                "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                "unit": UNIT,
                "path_labels": path_labels,
                "path_codes": path_labels[:],
                "path_ord": path_ord,
                "line_item": base_item_clean,   # ★ 괄호 제거된 정제본
                "line_index": line_idx,
                "group_id": sec_root_id, "parent_id": parent_id,
                "year_anchor": year_anchor, "statement_anchor": statement_anchor,
                "note_refs": note_refs,
                "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
            },
            "values": values,
            "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
        })

    return records


# ===== 여러 파일을 하나의 JSONL로 누적 저장 =====
def process_many(
    in_dir: Path,
    year_by_name_func,
    out_jsonl: Path,
    company: str,
) -> Path:
    in_dir = Path(in_dir)
    out_jsonl = Path(out_jsonl)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    htmls = sorted([p for p in in_dir.rglob("*") if p.suffix.lower() in {".htm", ".html"}])

    with out_jsonl.open("w", encoding="utf-8") as fw:
        for hp in htmls:
            try:
                y = year_by_name_func(hp)
            except Exception:
                y = 2014
            recs = extract_income_statement_records(hp, y, company)

            recs.sort(key=lambda r: (
                r.get("metadata", {}).get("year", 0),
                tuple(r.get("metadata", {}).get("path_ord", [])),
                r.get("id", "")
            ))

            for rec in recs:
                fw.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return out_jsonl


# ===== 실행 스니펫(모든 연도 → 하나의 JSONL) =====
def infer_year_from_name(p: Path) -> int:
    m = re.search(r"(19|20)\d{2}", p.stem)
    if not m:
        raise ValueError(f"연도 추출 실패: {p.name}")
    return int(m.group(0))

def run_folder_to_one_jsonl():
    IN_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
    OUT_JSONL = IN_DIR / "sec2_2_손익계산서.jsonl"

    test_file = IN_DIR / "감사보고서_2014.htm"
    if test_file.exists():
        _recs = extract_income_statement_records(test_file, 2014, "삼성전자")
        print(f"[TEST] 2014 레코드 수: {len(_recs)}")

    try:
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
    except OSError as e:
        print(f"[WARN] {e}; 홈으로 저장 시도")
        OUT_JSONL_HOME = Path.home() / "sec2_2_손익계산서.jsonl"
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL_HOME,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")

    # (선택) 프리뷰
    try:
        with out_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                print(line.rstrip()[:200])
    except Exception:
        pass


# ===== CLI =====
if __name__ == "__main__":
    if len(sys.argv) >= 4:
        in_arg   = Path(sys.argv[1])
        out_jsonl = Path(sys.argv[2])
        company  = sys.argv[3]
        year     = int(sys.argv[4]) if len(sys.argv) >= 5 else None

        if in_arg.is_dir():
            def _infer_year(p: Path) -> int:
                m = re.search(r"(19|20)\d{2}", p.stem)
                return int(m.group(0)) if m else (year or 2014)
            out_path = process_many(
                in_arg,
                year_by_name_func=_infer_year,
                out_jsonl=out_jsonl,
                company=company,
            )
            print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
        else:
            y = year
            if y is None:
                m = re.search(r"(19|20)\d{2}", in_arg.stem)
                y = int(m.group(0)) if m else 2014
            recs = extract_income_statement_records(in_arg, y, company)
            out_jsonl.parent.mkdir(parents=True, exist_ok=True)
            with out_jsonl.open("w", encoding="utf-8") as fw:
                for rec in recs:
                    fw.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"[OK] Wrote: {out_jsonl}  ({out_jsonl.stat().st_size} bytes)")
    else:
        run_folder_to_one_jsonl()

[OK] Wrote: /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec2_income_statement_all_years.jsonl  (123263 bytes)
{"id": "sec2-2014-is-매출액-매출액", "document": "2014년 삼성전자의 매출액은(는) 137,825,547 백만원입니다(전기 158,372,089 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "손익계산서", "statem
{"id": "sec2-2014-is-매출원가-매출원가", "document": "2014년 삼성전자의 매출원가은(는) 99,188,713 백만원입니다(전기 110,731,528 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "손익계산서", "stat
{"id": "sec2-2014-is-매출총이익-매출총이익", "document": "2014년 삼성전자의 매출총이익은(는) 38,636,834 백만원입니다(전기 47,640,561 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "손익계산서", "st
{"id": "sec2-2014-is-매출총이익-판매비와관리비", "document": "2014년의 삼성전자 매출총이익 중 하나인 판매비와관리비은(는) 24,711,840 백만원입니다(전기 25,833,556 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statemen
{"id": "sec2-2014-is-영업이익-영업이익", "document": "2014년 삼성전자의 영업이익은(는)

---

### 포괄손익계산서

In [ ]:
# -*- coding: utf-8 -*-
"""
SEC2 (포괄손익계산서) 파서 → JSONL 생성기 (로마숫자 항목만 추출, 문장에서는 로마숫자 제거)
- 대상 표: <table class="TABLE"> 중 포괄손익(당기순이익/기타포괄손익/총포괄손익) 키워드 포함
- 로마 숫자(Ⅰ. Ⅱ. Ⅲ. …)로 시작하는 행만 추출
- document 문장에서는 로마 숫자 접두를 제거해서 자연스럽게 출력
- 2014년에만 전기(prior) 값 포함
"""

from pathlib import Path
from bs4 import BeautifulSoup
import re, json

SECTION = "SEC2"
STATEMENT_NAME = "포괄손익계산서"
STATEMENT_CODE = "ci"
UNIT = "KRW_million"

RE_WS = re.compile(r"[ \t\u00A0]+")
RE_ROMAN_PREFIX = re.compile(r"^\s*((?:[IVXLCDM]+|[\u2160-\u2188]+))\.\s*")
RE_PARENS = re.compile(r"\([^)]*\)")
RE_INT = re.compile(r"^-?\d+$")
RE_DASH_ONLY = re.compile(r"^\s*[–—-]\s*$")

ROMAN_ASCII_MAP = {
    "Ⅰ":"I","Ⅱ":"II","Ⅲ":"III","Ⅳ":"IV","Ⅴ":"V","Ⅵ":"VI",
    "Ⅶ":"VII","Ⅷ":"VIII","Ⅸ":"IX","Ⅹ":"X","Ⅺ":"XI","Ⅻ":"XII"
}

def nrm(s: str) -> str:
    if s is None: return ""
    s = s.replace("\xa0", " ")
    return RE_WS.sub(" ", s).strip()

def strip_parens(s: str) -> str:
    return nrm(RE_PARENS.sub("", s or ""))

def roman_prefix(s: str):
    m = RE_ROMAN_PREFIX.match(s or "")
    if not m: return None, None
    raw = m.group(1)
    ascii_ = "".join(ROMAN_ASCII_MAP.get(ch, ch) for ch in raw)
    return raw, ascii_

def remove_roman_prefix(s: str) -> str:
    """문장/표시용: 'Ⅰ. 항목명' → '항목명'"""
    return RE_ROMAN_PREFIX.sub("", s or "").strip()

def slugify(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[ ]+", "-", s)
    s = re.sub(r"[^a-z0-9\-가-힣]", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s[:140] or "x"

def parse_number(x: str):
    s = nrm(x).replace(",", "")
    if not s or s.upper() == "NA" or RE_DASH_ONLY.match(s):
        return None
    if RE_INT.match(s):
        try:
            return int(s)
        except:
            return None
    return None

def fmt_int(num: int | None) -> str:
    return f"{num:,}" if num is not None else ""

def find_comprehensive_income_table(soup: BeautifulSoup):
    """
    <table class="TABLE"> 중 포괄손익 문맥(당기순이익/기타포괄손익/총포괄손익) 포함하는 표만 선택.
    여러 개인 경우 첫 번째로 일치하는 표 사용.
    """
    tables = soup.find_all("table", class_="TABLE")
    for t in tables:
        txt = nrm(t.get_text(" "))
        if (("당기순이익" in txt and "기타포괄손익" in txt) or ("총포괄손익" in txt)):
            return t
    return None

def extract_ci_records(html_path: Path, year: int, company: str):
    soup = BeautifulSoup(html_path.read_bytes(), "lxml")
    target = find_comprehensive_income_table(soup)
    if target is None:
        return []

    comp_key = slugify(company)
    year_anchor = f"{comp_key}-{year}"
    statement_anchor = f"{year_anchor}-{STATEMENT_CODE}"
    sec_root_id = f"sec2-{year}-{STATEMENT_CODE}-root"

    prev_year_effective = (year == 2014)
    records = []

    for tr in target.find_all("tr"):
        cells = tr.find_all(["th","td"])
        if len(cells) < 2:
            continue

        raw_label = nrm(cells[0].get_text(" "))
        raw_roman, _ = roman_prefix(raw_label)
        if not raw_roman:
            # 로마숫자 없는 행은 스킵
            continue

        texts = [nrm(c.get_text(" ")) for c in cells]

        # 일반적으로 [label, notes, cur_l, cur_r, prv_l, prv_r] 구조가 잦음
        cur = parse_number(texts[3]) if len(texts) > 3 else None
        prv = parse_number(texts[5]) if len(texts) > 5 else None
        if cur is None and len(texts) > 2:
            cur = parse_number(texts[2])
        if prv is None and len(texts) > 4:
            prv = parse_number(texts[4])
        if not prev_year_effective:
            prv = None

        # 표시는 괄호 제거 + 로마 접두 제거 버전 사용
        display_label_full = strip_parens(raw_label)
        display_label_noroman = remove_roman_prefix(display_label_full)

        rec_id = f"sec2-{year}-{STATEMENT_CODE}-{slugify(display_label_noroman)}"
        values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}

        # 문장(document)에서도 로마숫자 제거 버전 사용
        doc = f"{year}년 {company}의 {display_label_noroman}은(는) "
        if cur is not None:
            doc += f"{fmt_int(cur)} 백만원입니다"
            if prv is not None:
                doc += f"(전기 {fmt_int(prv)} 백만원)."
            else:
                doc += "."
        else:
            doc += "제공되지 않습니다."

        records.append({
            "id": rec_id,
            "document": doc,
            "metadata": {
                "company": company, "year": year,
                "section": SECTION,
                "statement": STATEMENT_NAME,
                "statement_code": STATEMENT_CODE,
                "unit": UNIT,
                "path_labels": [display_label_noroman],
                "path_codes": [display_label_noroman],
                "path_ord": [1, 1, 1],
                "line_item": display_label_noroman,
                "group_id": sec_root_id,
                "parent_id": sec_root_id,
                "year_anchor": year_anchor,
                "statement_anchor": statement_anchor,
            },
            "values": values,
            "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
        })

    return records

# ===== 실행 스니펫 =====
def infer_year_from_name(p: Path) -> int:
    m = re.search(r"(19|20)\d{2}", p.stem)
    return int(m.group(0)) if m else 2014

def run_folder_to_one_jsonl():
    IN_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
    OUT_JSONL = IN_DIR / "sec2_3_포괄손익계산서.jsonl"

    htmls = sorted([p for p in IN_DIR.glob("*.htm*")])
    with OUT_JSONL.open("w", encoding="utf-8") as fw:
        for hp in htmls:
            y = infer_year_from_name(hp)
            recs = extract_ci_records(hp, y, "삼성전자")
            for rec in recs:
                fw.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"[OK] Wrote: {OUT_JSONL} ({OUT_JSONL.stat().st_size} bytes)")

if __name__ == "__main__":
    run_folder_to_one_jsonl()


[OK] Wrote: /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec2_comprehensive_income_ROMAN.jsonl (23139 bytes)


---

### 자본변동표

In [ ]:
# -*- coding: utf-8 -*-
"""
SEC2 (자본변동표) 파서 → JSONL 생성기  [3번째 표]

- 각 '데이터 셀(행×열)'을 1 레코드로 생성
- line_item_raw 없음
- id는 기간 접미사(-p2013/-p2014 등)를 붙이지 않음
- 기간 정보는 metadata에만 기록: period_year / period_kind(전기|당기) / period_block("2013전기" 등)
"""
from pathlib import Path
from bs4 import BeautifulSoup
import re, json, sys

# ===== 섹션/명칭/코드 =====
SECTION = "SEC2"
STATEMENT_NAME = "자본변동표"
STATEMENT_CODE = "se"
UNIT = "KRW_million"

# ===== 정규식 유틸 =====
RE_WS = re.compile(r"[ \t\u00A0]+")
RE_DASH_ONLY = re.compile(r"^\s*[–—-]\s*$")
RE_INT = re.compile(r"^-?\d+$")
RE_NOTESPLIT = re.compile(r"[,\s]+")
RE_ROMAN_PREFIX = re.compile(r"^\s*((?:[IVXLCDM]+|[\u2160-\u2188]+))\.\s*")
RE_NUM_PREFIX = re.compile(r"^\s*(\d+)[\.\)]?\s*")
RE_PARENS = re.compile(r"\([^)]*\)")
# 제46기 / 제 46 (당)기 / 제45(전)기 / 제 45 기
RE_PERIOD_COL = re.compile(r"제\s*\d+\s*(?:\(\s*당\s*\)\s*기|\(\s*전\s*\)\s*기|기)\s*$")

# 행 라벨이 'YYYY.M.D (전기초|전기말|당기초|당기말)' 형태 (공백 변화 허용)
RE_DATE_MARK = re.compile(
    r"^(?P<y>20\d{2}|19\d{2})\s*\.\s*\d{1,2}\s*\.\s*\d{1,2}\s*\(\s*(?P<kind>전기초|전기말|당기초|당기말)\s*\)\s*$"
)

HEADER_TOKENS = {"과목", "항목", "구분", "주석"}

# 자본변동표에서 기대되는 '열(자본구성요소)' 키
EXPECTED_COL_KEYS = {
    "자본금", "주식발행초과금", "이익잉여금", "기타자본항목", "매각예정분류기타자본항목", "총계"
}

# ===== 문자열/정규화 유틸 =====
def nrm(s: str) -> str:
    if s is None: return ""
    s = s.replace("\xa0", " ")
    return RE_WS.sub(" ", s).strip()

def strip_roman_prefix(s: str) -> str:
    return RE_ROMAN_PREFIX.sub("", s or "").strip()

def strip_num_prefix(s: str) -> str:
    return RE_NUM_PREFIX.sub("", s or "").strip()

def parse_line_number(s: str):
    if not s: return None
    m = RE_NUM_PREFIX.match(s)
    if not m: return None
    try: return int(m.group(1))
    except: return None

def strip_parens(s: str) -> str:
    return nrm(RE_PARENS.sub("", s or ""))

def collapse_hangul_letters(s: str) -> str:
    t = nrm(s)
    if not t: return t
    return re.sub(r"[ \t·\-\.\u00B7]+", "", t)

def clean_label(s: str) -> str:
    # 로마 접두 제거 → 한글 글자 결합 → 공백정리
    return nrm(collapse_hangul_letters(strip_roman_prefix(s or "")))

def slugify(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[ ]+", "-", s)
    s = re.sub(r"[^a-z0-9\-가-힣]", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s[:140] or "x"

def parse_number(x: str):
    s = nrm(x).replace(",", "")
    if not s or s.upper()=="NA" or RE_DASH_ONLY.match(s): return None
    if RE_INT.match(s):
        try: return int(s)
        except: return None
    return None

def split_notes(txt: str):
    s = nrm(txt)
    if not s: return []
    parts = [p.strip("()[]") for p in RE_NOTESPLIT.split(s) if p]
    out = []
    for p in parts:
        p2 = p.replace(" ", "")
        if re.fullmatch(r"\d+(?:\.\d+)?", p2):
            out.append(p2)
    return out

def company_anchor(name: str) -> str:
    base = re.sub(r"[^\w\s-]", "", nrm(name)).strip().lower()
    base = re.sub(r"\s+", "-", base)
    return slugify(base) or "company"

def fmt_int(num: int | None) -> str:
    if num is None: return ""
    return f"{num:,}"

def normalize_col_name(s: str) -> str:
    x = strip_parens(clean_label(s))
    # 대표 변형 정규화
    if x.replace(" ", "") in {"총계", "총  계", "총   계"}:
        return "총계"
    return x

# ===== 표 선택/스코어 =====
def _extract_header_cells(table):
    """thead가 있으면 마지막 thead tr만, 없으면 tbody 첫 1~2행에서 헤더 추정"""
    thead = table.find("thead")
    if thead:
        trs = thead.find_all("tr")
        if trs:
            last_tr = trs[-1]
            return [nrm(th.get_text(" ")) for th in last_tr.find_all(["th","td"])]
    # 폴백: tbody 첫 행(들)
    tbody = table.find("tbody")
    if tbody:
        trs = tbody.find_all("tr", limit=2)
        if trs:
            return [nrm(x.get_text(" ")) for x in trs[-1].find_all(["th","td"])]
    return []

def _score_table_for_se(table):
    """예상 열 키워드 매칭 개수로 점수화 (자본변동표용)"""
    headers = _extract_header_cells(table)
    if len(headers) < 3:
        return -1, [], []
    raw_cols = headers[2:]  # 앞 2칸: 과목, 주석 가정
    norm_cols = [normalize_col_name(h) for h in raw_cols]
    # 기간열 제외
    data_cols = [c for r,c in zip(raw_cols, norm_cols) if not RE_PERIOD_COL.search(r.replace(" ", ""))]
    # 기대 키워드 매칭 수
    hits = sum(1 for c in data_cols if c in EXPECTED_COL_KEYS)
    return hits, raw_cols, data_cols

def _pick_se_table(soup):
    """가장 그럴듯한 자본변동표 표 선택"""
    candidates = []
    for t in soup.find_all("table"):
        cls = " ".join((t.get("class") or [])).lower()
        if "table" not in cls:
            continue
        headers = _extract_header_cells(t)
        if not headers:
            continue
        header_text = " ".join(headers)
        # 머리말에 '과목/항목/구분/주석' 등이 보이면 후보
        if any(tok in header_text for tok in HEADER_TOKENS):
            score, raw_cols, data_cols = _score_table_for_se(t)
            candidates.append((score, t, raw_cols, data_cols))

    if not candidates:
        return None, [], []

    # 1) 기대 열 키워드 매칭 가장 높은 표
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score = candidates[0][0]
    if best_score >= 2:  # 최소 2개 이상 매칭
        return candidates[0][1], candidates[0][2], candidates[0][3]

    # 2) 매칭이 빈약하면, 3번째(0-index 2) → 마지막 순으로 폴백
    tables_only = [t for _,t,_,_ in candidates]
    if len(tables_only) >= 3:
        sc, rc, dc = _score_table_for_se(tables_only[2])
        return tables_only[2], rc, dc
    return tables_only[-1], candidates[-1][2], candidates[-1][3]

# ===== 핵심 파서 =====
def extract_equity_changes_table(html_path: Path, year: int, company: str):
    soup = BeautifulSoup(html_path.read_bytes(), "lxml")

    target, raw_cols, data_cols = _pick_se_table(soup)
    if target is None or not data_cols:
        return []

    comp_key = company_anchor(company)
    year_anchor = f"{comp_key}-{year}"
    statement_anchor = f"{year_anchor}-{STATEMENT_CODE}"
    sec_root_id = f"sec2-{year}-{STATEMENT_CODE}-root"

    records = []
    current_mid = None                     # 로마 섹션(Ⅰ. …)
    current_period_year = None             # 2013 / 2014 ...
    current_period_kind = None             # "전기" / "당기"

    def make_id(mid_label, row_item_disp, col_name):
        base = "root" if not mid_label else slugify(mid_label)
        num = parse_line_number(row_item_disp or "")
        row_clean = strip_parens(strip_num_prefix(row_item_disp or ""))
        tail = slugify((str(num)+"-"+row_clean) if num is not None else row_clean)
        col_clean = slugify(strip_parens(col_name or ""))
        # ⛔ 기간 접미사(-p2013 등) 절대 추가하지 않음
        return f"sec2-{year}-{STATEMENT_CODE}-{base}-{tail}-{col_clean}"

    def path_ord(mid_label, row_item_disp):
        out = [1]
        if not hasattr(path_ord, "_mid_map"):
            path_ord._mid_map = {}; path_ord._next = 1
        if mid_label:
            if mid_label not in path_ord._mid_map:
                path_ord._mid_map[mid_label] = path_ord._next; path_ord._next += 1
            out.append(path_ord._mid_map[mid_label])
        if not hasattr(path_ord, "_row_counter"):
            path_ord._row_counter = {}
        key = mid_label or "_root"
        if parse_line_number(row_item_disp or "") is not None:
            out.append(parse_line_number(row_item_disp or ""))
        else:
            path_ord._row_counter.setdefault(key, 0)
            path_ord._row_counter[key] += 1
            out.append(path_ord._row_counter[key])
        return out

    def update_period_by_rowlabel(row_label_raw: str):
        """행 라벨이 'YYYY.MM.DD (전/당기초/말)'일 때 기간컨텍스트 갱신"""
        nonlocal current_period_year, current_period_kind
        m = RE_DATE_MARK.match(nrm(row_label_raw))
        if not m: return
        y = int(m.group("y"))
        kind = m.group("kind")  # 전기초/전기말/당기초/당기말
        # kind를 전기/당기로 축약
        kind_simple = "전기" if kind.startswith("전기") else "당기"
        current_period_year = y
        current_period_kind = kind_simple

    # tbody rows
    tbody = target.find("tbody") or target
    rows = tbody.find_all("tr")
    for tr in rows:
        tds = tr.find_all("td")
        if len(tds) < 3:
            continue

        row_label_raw = nrm(tds[0].get_text(" "))
        notes_raw = nrm(tds[1].get_text(" "))

        # 기간 컨텍스트 업데이트 (기초/기말 행을 만나면 현재 블록 설정)
        update_period_by_rowlabel(row_label_raw)

        # 섹션(로마 숫자) 행
        if RE_ROMAN_PREFIX.match(row_label_raw):
            current_mid = clean_label(row_label_raw)
            continue

        # 날짜행(전기초/말, 당기초/말)은 보통 합계 스냅샷이므로 여기선 스킵
        if RE_DATE_MARK.match(row_label_raw):
            continue

        # 일반 행
        row_item_clean = clean_label(row_label_raw)
        if not row_item_clean:
            continue
        note_refs = split_notes(notes_raw)

        # tds[2:]와 raw_cols는 1:1 매핑 → 기간열은 스킵하고 data_cells 구성
        data_cells = []
        for raw, cell in zip(raw_cols, tds[2:]):
            if RE_PERIOD_COL.search(raw.replace(" ", "")):
                continue
            data_cells.append(cell)

        L = min(len(data_cols), len(data_cells))
        for j in range(L):
            col_name = data_cols[j]
            val = parse_number(nrm(data_cells[j].get_text(" ")))
            if val is None:
                continue

            mid = current_mid
            row_label_for_path = strip_parens(strip_num_prefix(row_item_clean))
            p_labels = [mid, row_label_for_path] if mid else [row_label_for_path]
            rec_id = make_id(mid, row_item_clean, col_name)

            # document 문장 (기간 정보 반영)
            if current_period_year and current_period_kind:
                if mid:
                    doc = f"{current_period_year}년 {current_period_kind} {company}의 {mid} 중 '{p_labels[-1]}'에 대한 {col_name} 금액은 {fmt_int(val)} 백만원입니다."
                else:
                    doc = f"{current_period_year}년 {current_period_kind} {company}의 '{p_labels[-1]}'에 대한 {col_name} 금액은 {fmt_int(val)} 백만원입니다."
            else:
                # 기간컨텍스트 못 찾은 경우: 기본 문장
                if mid:
                    doc = f"{year}년 {company}의 {mid} 중 '{p_labels[-1]}'에 대한 {col_name} 금액은 {fmt_int(val)} 백만원입니다."
                else:
                    doc = f"{year}년 {company}의 '{p_labels[-1]}'에 대한 {col_name} 금액은 {fmt_int(val)} 백만원입니다."

            meta = {
                "company": company, "year": year,
                "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                "unit": UNIT,
                "path_labels": p_labels,
                "path_codes": p_labels[:],
                "path_ord": path_ord(mid, row_item_clean),
                "line_item": strip_parens(col_name),
                "line_index": parse_line_number(row_item_clean),
                "group_id": sec_root_id,
                "parent_id": sec_root_id,
                "year_anchor": f"{company_anchor(company)}-{year}",
                "statement_anchor": statement_anchor,
                "note_refs": note_refs,
                "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
            }
            # 기간 메타데이터(있을 때만)
            if current_period_year:
                meta["period_year"] = int(current_period_year)
            if current_period_kind:
                meta["period_kind"] = current_period_kind
                meta["period_block"] = f"{current_period_year}{current_period_kind}"

            records.append({
                "id": rec_id,
                "document": doc,
                "metadata": meta,
                "values": {"current": val},
                "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
            })

    return records

# ====== 배치 실행 유틸 ======
def process_many(in_dir: Path, year_by_name_func, out_jsonl: Path, company: str) -> Path:
    in_dir = Path(in_dir)
    out_jsonl = Path(out_jsonl)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    htmls = sorted([p for p in in_dir.rglob("*") if p.suffix.lower() in {".htm", ".html"}])

    with out_jsonl.open("w", encoding="utf-8") as fw:
        for hp in htmls:
            try:
                y = year_by_name_func(hp)
            except Exception:
                y = 2014
            recs = extract_equity_changes_table(hp, y, company)

            recs.sort(key=lambda r: (
                r.get("metadata", {}).get("year", 0),
                tuple(r.get("metadata", {}).get("path_ord", [])),
                r.get("id", "")
            ))
            for rec in recs:
                fw.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return out_jsonl

# ====== 실행 스니펫 ======
def infer_year_from_name(p: Path) -> int:
    m = re.search(r"(19|20)\d{2}", p.stem)
    if not m:
        raise ValueError(f"연도 추출 실패: {p.name}")
    return int(m.group(0))

def run_folder_to_one_jsonl():
    IN_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
    OUT_JSONL = IN_DIR / "sec2_3_자본변동표.jsonl"

    test_file = IN_DIR / "감사보고서_2014.htm"
    if test_file.exists():
        _recs = extract_equity_changes_table(test_file, 2014, "삼성전자")
        print(f"[TEST] 2014 레코드 수: {len(_recs)}")

    try:
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
    except OSError as e:
        print(f"[WARN] {e}; 홈으로 저장 시도")
        OUT_JSONL_HOME = Path.home() / "sec2_3_자본변동표.jsonl"
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL_HOME,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")

    try:
        with out_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                print(line.rstrip()[:220])
    except Exception:
        pass

# ====== CLI ======
if __name__ == "__main__":
    if len(sys.argv) >= 4:
        in_arg   = Path(sys.argv[1])
        out_jsonl = Path(sys.argv[2])
        company  = sys.argv[3]
        year     = int(sys.argv[4]) if len(sys.argv) >= 5 else None

        if in_arg.is_dir():
            def _infer_year(p: Path) -> int:
                m = re.search(r"(19|20)\d{2}", p.stem)
                return int(m.group(0)) if m else (year or 2014)
            out_path = process_many(
                in_arg,
                year_by_name_func=_infer_year,
                out_jsonl=out_jsonl,
                company=company,
            )
            print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
        else:
            y = year
            if y is None:
                m = re.search(r"(19|20)\d{2}", in_arg.stem)
                y = int(m.group(0)) if m else 2014
            recs = extract_equity_changes_table(in_arg, y, company)
            out_jsonl.parent.mkdir(parents=True, exist_ok=True)
            with out_jsonl.open("w", encoding="utf-8") as fw:
                for rec in recs:
                    fw.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"[OK] Wrote: {out_jsonl}  ({out_jsonl.stat().st_size} bytes)")
    else:
        run_folder_to_one_jsonl()

[OK] Wrote: /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec2_equity_changes_all_years.jsonl  (276453 bytes)
{"id": "sec2-2014-se-총포괄손익-1-당기순이익-이익잉여금", "document": "2013년 전기 삼성전자의 총포괄손익 중 '당기순이익'에 대한 이익잉여금 금액은 17,929,520 백만원입니다.", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "자본변동표", "statement_
{"id": "sec2-2014-se-총포괄손익-1-당기순이익-이익잉여금", "document": "2014년 당기 삼성전자의 총포괄손익 중 '당기순이익'에 대한 이익잉여금 금액은 14,591,781 백만원입니다.", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "자본변동표", "statement_
{"id": "sec2-2014-se-총포괄손익-1-당기순이익-총계", "document": "2013년 전기 삼성전자의 총포괄손익 중 '당기순이익'에 대한 총계 금액은 17,929,520 백만원입니다.", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "자본변동표", "statement_code":
{"id": "sec2-2014-se-총포괄손익-1-당기순이익-총계", "document": "2014년 당기 삼성전자의 총포괄손익 중 '당기순이익'에 대한 총계 금액은 14,591,781 백만원입니다.", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "자본변동표", "state

---

### 현금흐름표

In [ ]:
# -*- coding: utf-8 -*-
"""
SEC2 (현금흐름표) 파서 → JSONL 생성기

- SEC2의 4번째 <table class="TABLE"> (혹은 헤더 키워드 스코어)에서 파싱
- 각 '데이터 셀(행×열)'을 1 레코드로 생성
- 로마 숫자 섹션(I~VI)은 '중간 섹션'으로 취급해 자체 값이 있으면 레코드 생성
- id: 숫자 접두어 존재 시 '-<숫자>-<라벨>'로 분리
- metadata.line_item: 숫자 제거된 라벨, line_item_raw: 원문 보존 없음(필요시 추가)
- 2014년에만 prior(전기값) 유지
"""
from pathlib import Path
from bs4 import BeautifulSoup
import re, json, sys

# ===== 공통 설정 =====
SECTION = "SEC2"
STATEMENT_NAME = "현금흐름표"
STATEMENT_CODE = "cf"
UNIT = "KRW_million"

# ===== 정규식 / 헬퍼 =====
RE_WS = re.compile(r"[ \t\u00A0]+")
RE_DASH_ONLY = re.compile(r"^\s*[–—-]\s*$")
RE_INT = re.compile(r"^-?\d+$")
RE_NOTESPLIT = re.compile(r"[,\s]+")
RE_ROMAN_PREFIX = re.compile(r"^\s*((?:[IVXLCDM]+|[\u2160-\u2188]+))\.\s*")
RE_NUM_PREFIX = re.compile(r"^\s*(\d+|[가-하])[\.\)]?\s*")   # 1. 2. 3. / 가. 나. 다. 모두 허용

ROMAN_ASCII_MAP = {
    "Ⅰ":"I","Ⅱ":"II","Ⅲ":"III","Ⅳ":"IV","Ⅴ":"V","Ⅵ":"VI","Ⅶ":"VII","Ⅷ":"VIII","Ⅸ":"IX","Ⅹ":"X",
    "Ⅺ":"XI","Ⅻ":"XII","Ⅼ":"L","Ⅽ":"C","Ⅾ":"D","Ⅿ":"M"
}

HEADER_TOKENS = {"과목", "항목", "구분", "주석"}

def nrm(s: str) -> str:
    if s is None: return ""
    s = s.replace("\xa0", " ")
    return RE_WS.sub(" ", s).strip()

def collapse_hangul_letters(s: str) -> str:
    t = nrm(s)
    if not t: return t
    return re.sub(r"[ \t·\-\.\u00B7]+", "", t)

def strip_roman_prefix(s: str) -> str:
    return RE_ROMAN_PREFIX.sub("", s or "").strip()

def strip_num_prefix(s: str) -> str:
    return RE_NUM_PREFIX.sub("", s or "").strip()

def parse_line_number(s: str):
    if not s: return None
    m = RE_NUM_PREFIX.match(s)
    if not m: return None
    tok = m.group(1)
    try:
        # 가/나/다 … → 번호 대신 None 유지
        return int(tok)
    except Exception:
        return None

def roman_to_ascii(raw: str) -> str:
    return "".join(ROMAN_ASCII_MAP.get(ch, ch) for ch in raw)

def roman_prefix(s: str):
    m = RE_ROMAN_PREFIX.match(s or "")
    if not m:
        return None, None
    raw = m.group(1)
    return raw, roman_to_ascii(raw)

def slugify(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[ ]+", "-", s)
    s = re.sub(r"[^a-z0-9\-가-힣]", "-", s)
    s = re.sub(r"-{2,}", "-", s).strip("-")
    return s[:140] or "x"

def parse_number(x: str):
    s = nrm(x).replace(",", "")
    if not s or s.upper()=="NA" or RE_DASH_ONLY.match(s): return None
    if RE_INT.match(s):
        try: return int(s)
        except: return None
    return None

def split_notes(txt: str):
    s = nrm(txt)
    if not s: return []
    parts = [p.strip("()[]") for p in RE_NOTESPLIT.split(s) if p]
    out = []
    for p in parts:
        p2 = p.replace(" ", "")
        if re.fullmatch(r"\d+(?:\.\d+)?", p2):
            out.append(p2)
    return out

def company_anchor(name: str) -> str:
    base = re.sub(r"[^\w\s-]", "", nrm(name)).strip().lower()
    base = re.sub(r"\s+", "-", base)
    return slugify(base) or "company"

def fmt_int(num: int | None) -> str:
    if num is None: return ""
    return f"{num:,}"

# ===== 정렬용 =====
MID_ORD = {"영업활동현금흐름":1, "투자활동현금흐름":2, "재무활동현금흐름":3,
           "현금및현금성자산의감소(Ⅰ+Ⅱ+Ⅲ)":4, "기초의현금및현금성자산":5, "기말의현금및현금성자산":6}

def get_mid_ord(mid: str) -> int:
    return MID_ORD.get(mid.replace(" ", ""), 99)

# ===== 테이블 선택 =====
def _pick_cf_table(soup):
    tables = soup.find_all("table")
    # 1) “영업활동 현금흐름” 키워드 스코어
    scored = []
    for i, t in enumerate(tables):
        txt = nrm(t.get_text(" "))
        score = sum(k in txt for k in ["영업활동 현금흐름","투자활동 현금흐름","재무활동 현금흐름"])
        scored.append((score, i, t))
    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)
    if scored and scored[0][0] >= 2:
        return scored[0][2]

    # 2) 폴백: SEC2의 4번째 table(class/TABLE 무관)
    if len(tables) >= 4:
        return tables[3]
    return None

# ===== 핵심 파서 =====
def extract_cashflow_records(html_path: Path, year: int, company: str):
    soup = BeautifulSoup(html_path.read_bytes(), "lxml")
    target = _pick_cf_table(soup)
    if target is None:
        return []

    comp_key = company_anchor(company)
    year_anchor = f"{comp_key}-{year}"
    statement_anchor = f"{year_anchor}-{STATEMENT_CODE}"
    sec_root_id = f"sec2-{year}-{STATEMENT_CODE}-root"

    current_mid = None  # I/II/III/IV/V/VI …
    line_ord_counter = {}

    def make_record_id(code_path, line_item_disp):
        base = "-".join(code_path) if code_path else "root"
        num = parse_line_number(line_item_disp or "")
        li = strip_num_prefix(line_item_disp or "")
        if num is not None:
            return f"sec2-{year}-{STATEMENT_CODE}-{slugify(base + '-' + str(num) + '-' + li)}"
        else:
            return f"sec2-{year}-{STATEMENT_CODE}-{slugify(base + '-' + li)}"

    def get_path_ord(path_labels, line_item_display=None):
        out = []
        if path_labels:
            mid = path_labels[0] if len(path_labels)==1 else path_labels[1]
            out.append(get_mid_ord(mid))
        if line_item_display is not None:
            num = parse_line_number(line_item_display or "")
            if num is not None:
                out.append(num)
            else:
                key = tuple(path_labels)
                line_ord_counter.setdefault(key, 0)
                line_ord_counter[key] += 1
                out.append(line_ord_counter[key])
        return out

    def clean_display_label(s: str) -> str:
        t = strip_roman_prefix(s or "")
        t = collapse_hangul_letters(t)
        return nrm(t)

    # “제 46(당)기 / 제45(전)기”는 각 2칸(colspan=2). → 각 페어에서 값이 있는 쪽을 채택.
    def take_pair(v1, v2):
        a = parse_number(v1)
        b = parse_number(v2)
        return a if a is not None else b

    rows = target.find_all("tr")
    prev_year_effective = (year == 2014)
    records = []

    for tr in rows:
        cells = tr.find_all(["th","td"])
        if len(cells) < 2:  # 라벨조차 없으면 스킵
            continue

        texts = [nrm(c.get_text(" ")) for c in cells]
        first = texts[0].replace(" ", "")
        if first in HEADER_TOKENS:
            continue

        raw_label = texts[0] if len(texts)>=1 else ""
        raw_notes = texts[1] if len(texts)>=2 else ""

        # [label, notes, cL, cR, pL, pR]
        cur = take_pair(texts[2] if len(texts)>2 else "", texts[3] if len(texts)>3 else "")
        prv = take_pair(texts[4] if len(texts)>4 else "", texts[5] if len(texts)>5 else "")
        if not prev_year_effective:
            prv = None

        # 섹션 헤더(I., II., …)
        raw_roman, _ = roman_prefix(raw_label)
        if raw_roman:
            current_mid = clean_display_label(raw_label)  # 예: 영업활동현금흐름
            # 섹션 자체 값(합계)이 있으면 레코드 생성
            if (cur is not None) or (prv is not None):
                path_labels = ["현금흐름", current_mid]
                rec_id = make_record_id(path_labels, current_mid)
                note_refs = split_notes(raw_notes)
                values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}
                doc = (f"{year}년 {company}의 {current_mid}은(는) {fmt_int(cur)} 백만원입니다."
                       if prv is None else
                       f"{year}년 {company}의 {current_mid}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원).")
                records.append({
                    "id": rec_id,
                    "document": doc,
                    "metadata": {
                        "company": company, "year": year,
                        "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                        "unit": UNIT,
                        "path_labels": path_labels, "path_codes": path_labels[:],
                        "path_ord": get_path_ord(path_labels, None),
                        "line_item": current_mid,
                        "group_id": sec_root_id, "parent_id": sec_root_id,
                        "year_anchor": year_anchor, "statement_anchor": statement_anchor,
                        "note_refs": note_refs,
                        "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
                    },
                    "values": values,
                    "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
                })
            continue

        # 일반 라인(하위 항목)
        display_label = clean_display_label(raw_label)
        if not display_label:
            continue
        path_labels = (["현금흐름", current_mid] if current_mid else ["현금흐름"])
        base_item = strip_num_prefix(display_label)
        line_idx = parse_line_number(display_label)

        rec_id = make_record_id(path_labels, display_label)
        note_refs = split_notes(raw_notes)
        values = {"current": cur, "prior": prv} if prev_year_effective else {"current": cur}

        if cur is None and prv is None:
            doc = f"{year}년 {company}의 {('/'.join(path_labels)+' 중 ' if path_labels else '')}{base_item}은(는) 제공되지 않습니다."
        else:
            if prv is None:
                doc = f"{year}년 {company}의 {('/'.join(path_labels)+' 중 ' if path_labels else '')}{base_item}은(는) {fmt_int(cur)} 백만원입니다."
            else:
                doc = f"{year}년 {company}의 {('/'.join(path_labels)+' 중 ' if path_labels else '')}{base_item}은(는) {fmt_int(cur)} 백만원입니다(전기 {fmt_int(prv)} 백만원)."

        records.append({
            "id": rec_id,
            "document": doc,
            "metadata": {
                "company": company, "year": year,
                "section": SECTION, "statement": STATEMENT_NAME, "statement_code": STATEMENT_CODE,
                "unit": UNIT,
                "path_labels": path_labels, "path_codes": path_labels[:],
                "path_ord": get_path_ord(path_labels, display_label),
                "line_item": base_item, "line_index": line_idx,
                "group_id": sec_root_id, "parent_id": sec_root_id,
                "year_anchor": year_anchor, "statement_anchor": statement_anchor,
                "note_refs": note_refs,
                "note_doc_ids": [f"sec3-{year}-note-{n}" for n in note_refs] if note_refs else []
            },
            "values": values,
            "source": f"{html_path.name}#SEC2-{STATEMENT_NAME}"
        })

    return records


# ===== 여러 파일을 하나의 JSONL로 누적 저장 =====
def process_many(in_dir: Path, year_by_name_func, out_jsonl: Path, company: str) -> Path:
    in_dir = Path(in_dir)
    out_jsonl = Path(out_jsonl)
    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    htmls = sorted([p for p in in_dir.rglob("*") if p.suffix.lower() in {".htm", ".html"}])

    with out_jsonl.open("w", encoding="utf-8") as fw:
        for hp in htmls:
            try:
                y = year_by_name_func(hp)
            except Exception:
                y = 2014
            recs = extract_cashflow_records(hp, y, company)
            recs.sort(key=lambda r: (
                r.get("metadata", {}).get("year", 0),
                tuple(r.get("metadata", {}).get("path_ord", [])),
                r.get("id", "")
            ))
            for rec in recs:
                fw.write(json.dumps(rec, ensure_ascii=False) + "\n")

    return out_jsonl


# ===== 실행 스니펫 =====
def infer_year_from_name(p: Path) -> int:
    m = re.search(r"(19|20)\d{2}", p.stem)
    if not m:
        raise ValueError(f"연도 추출 실패: {p.name}")
    return int(m.group(0))

def run_folder_to_one_jsonl():
    IN_DIR = Path("/Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed")
    OUT_JSONL = IN_DIR / "sec2_5_현금흐름표.jsonl"

    test_file = IN_DIR / "감사보고서_2014.htm"
    if test_file.exists():
        _recs = extract_cashflow_records(test_file, 2014, "삼성전자")
        print(f"[TEST] 2014 레코드 수: {len(_recs)}")

    try:
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
    except OSError as e:
        print(f"[WARN] {e}; 홈으로 저장 시도")
        OUT_JSONL_HOME = Path.home() / "sec2_5_현금흐름표.jsonl"
        out_path = process_many(
            IN_DIR,
            year_by_name_func=infer_year_from_name,
            out_jsonl=OUT_JSONL_HOME,
            company="삼성전자",
        )
        print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")

    try:
        with out_path.open("r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 5: break
                print(line.rstrip()[:220])
    except Exception:
        pass


# ===== CLI =====
if __name__ == "__main__":
    if len(sys.argv) >= 4:
        in_arg   = Path(sys.argv[1])
        out_jsonl = Path(sys.argv[2])
        company  = sys.argv[3]
        year     = int(sys.argv[4]) if len(sys.argv) >= 5 else None

        if in_arg.is_dir():
            def _infer_year(p: Path) -> int:
                m = re.search(r"(19|20)\d{2}", p.stem)
                return int(m.group(0)) if m else (year or 2014)
            out_path = process_many(
                in_arg,
                year_by_name_func=_infer_year,
                out_jsonl=out_jsonl,
                company=company,
            )
            print(f"[OK] Wrote: {out_path}  ({out_path.stat().st_size} bytes)")
        else:
            y = year
            if y is None:
                m = re.search(r"(19|20)\d{2}", in_arg.stem)
                y = int(m.group(0)) if m else 2014
            recs = extract_cashflow_records(in_arg, y, company)
            out_jsonl.parent.mkdir(parents=True, exist_ok=True)
            with out_jsonl.open("w", encoding="utf-8") as fw:
                for rec in recs:
                    fw.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"[OK] Wrote: {out_jsonl}  ({out_jsonl.stat().st_size} bytes)")
    else:
        run_folder_to_one_jsonl()


[OK] Wrote: /Users/sungwoo/SNU/자연어처리_프로젝트/삼성전자_감사보고서_2014_2024/preprocessed/sec2_cashflow_all_years.jsonl  (312587 bytes)
{"id": "sec2-2014-cf-현금흐름-영업활동현금흐름-업활동현금흐름", "document": "2014년 삼성전자의 영업활동현금흐름은(는) 18,653,817 백만원입니다(전기 28,443,058 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "현금흐름표", "statement
{"id": "sec2-2014-cf-현금흐름-영업활동현금흐름-1-영업에서창출된현금흐름", "document": "2014년 삼성전자의 현금흐름/영업활동현금흐름 중 영업에서창출된현금흐름은(는) 20,854,601 백만원입니다(전기 31,520,938 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statem
{"id": "sec2-2014-cf-현금흐름-영업활동현금흐름-당기순이익", "document": "2014년 삼성전자의 현금흐름/영업활동현금흐름 중 당기순이익은(는) 14,591,781 백만원입니다(전기 17,929,520 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "현금흐름표",
{"id": "sec2-2014-cf-현금흐름-영업활동현금흐름-2-이자의수취", "document": "2014년 삼성전자의 현금흐름/영업활동현금흐름 중 이자의수취은(는) 854,946 백만원입니다(전기 441,659 백만원).", "metadata": {"company": "삼성전자", "year": 2014, "section": "SEC2", "statement": "현금흐름표"